# 🎧 AD-Qualitätsnotebook Pro (MDR ↔ KI) · Kurzbeschreibung

Dieses interaktive Notebook vergleicht zwei Audiodeskriptions-Texte **satzweise** –  
den **MDR-Referenztext** und einen **KI-generierten Hypothesentext** – und bewertet deren  
**inhaltliche, sprachliche und zeitliche Übereinstimmung** anhand objektiver Kennzahlen.

---

## ⚙️ Funktionsüberblick

- **Segmentierung & Alignment:** automatische Satz-Zuordnung über *TF-IDF + Cosine Similarity*,  
  optional erweitert um zeitbasiertes Matching (*time_aware_alignment*).
- **Kernmetriken:** BLEU-1 … 4, ROUGE-1/2/L, optional *SBERT Cosine* und *BERTScore F1*.
- **RAGAS-Kennzahlen:** *semantic_similarity*, *answer_similarity*, *faithfulness*,  
  *answer_relevancy*, *coverage*, *conciseness*, *fluency (FRE)* – mit Fallback-Berechnung, falls die RAGAS-Bibliothek nicht installiert ist.
- **Heuristiken:** Erkennung von **Farbangaben** und **Bewegungsverben**  
  (lemma-basiert über *spaCy* oder token-basiert im Fallback-Modus).
- **Längen- & Lesbarkeitsmaße:** Wort- und Silbenzahlen, Verhältnis-Indikatoren (*len_ratio*, *syl_ratio*),  
  sowie **Flesch–Amstad-Lesbarkeitsindex (DE)**.
- **Timing-Analyse:** Start-, End- und Mittel-Abweichungen (Δ-Werte), Zeit-Overlap,  
  *Timing Accuracy* und grafische Auswertungen (Histogramme & Scatterplots).
- **Darstellung:** interaktive Tabs  
  (*Text*, *Qualität*, *ROUGE*, *Länge & Abdeckung*, *Alignment*, *Timing*, optional *RAGAS*, *Alle*, *⚙️ Einstellungen*)  
  mit Sticky-Headern und Ampel-Visualisierung.
- **Reporting & Rubrics:** automatische Aggregation aller Ergebnisse in Tabellen, Diagrammen  
  und MDR-Rubrics (0 – 100 Punkte) inkl. Gesamtnote, Empfehlungen und Markdown-Export.

---

## 🧭 Hinweise zur Nutzung

- **Dateien laden:** MDR-Referenz und KI-Text (je `.txt`) hochladen → **Laden** → **Berechnen**.  
- **Ampel-Schwellen:** anpassbar im Tab *⚙️ Einstellungen* (`THRESHOLDS`-Parameter).  
- **Optionale Bibliotheken:** *SBERT*, *BERTScore*, *spaCy*, *RAGAS* werden nur verwendet, wenn verfügbar.  
- **Export:** automatische Markdown-Berichte und TSV/Excel-Dateien können direkt erzeugt werden.  
- **UI-Elemente:** alle Tabellen sind scrollbar, Kopfzeilen fixiert, Layout 72 vh hoch (optimiert für Jupyter).

---

> **Ziel:**  
> Das Qualitätsnotebook bietet ein reproduzierbares, metrisch fundiertes Verfahren zur  
> **automatisierten Bewertung von Audiodeskriptionen** nach linguistischen, semantischen  
> und zeitlichen Kriterien – einschließlich visueller und stilistischer Aspekte.


In [12]:
# OPTIONAL: Installieren, falls erforderlich
# %pip install -q sentence-transformers bert-score matplotlib pandas numpy

# !pip install ragas nest_asyncio sentence-transformers langchain-openai

# %pip install -U spacy
# !python -m spacy download de_core_news_sm

# import sys
# !{sys.executable} -m pip install -U pyphen


# 📊 Kennzahlen – Definition, Berechnung, Interpretation & Empfehlungen

Diese Seite erklärt **alle Metriken** des aktuellen *AD-Qualitätsnotebook Pro (v10)* – vom  
klassischen NLP-Score bis zu **Timing-, Rubric- und RAGAS-ähnlichen Kennzahlen** –  
in **einfacher Sprache** und mit **praxisnahen Empfehlungen** zur Qualitätsverbesserung.

> **Hinweise**
> - **Ampel-Schwellen** sind im Tab *⚙️ Einstellungen* interaktiv über `THRESHOLDS`-Slider anpassbar.  
> - **SBERT**, **BERTScore**, **spaCy** und **RAGAS** werden **nur verwendet, wenn installiert**.  
> - **RAGAS-Werte** werden segmentweise (*Ref ↔ Hyp*) berechnet; falls Bibliotheken fehlen,  
>   greift ein **Fallback** aus *ROUGE / SBERT / BERTScore / FRE*.  
> - **Timing-Metriken** basieren auf Start-, End- und Mittel-Δ-Werten, Overlap und Accuracy.  
> - **Mehrfach-Zuordnungen** werden automatisch bereinigt → nur der **beste Treffer** bleibt aktiv.

---

## 1️⃣ Klassische NLP-Metriken

| **Kennzahl** | **Beschreibung** | **Berechnung (vereinfacht)** | **Skala** | **Interpretation** | **Empfehlung** |
|:--------------|:-----------------|:------------------------------|:-----------|:-------------------|:---------------|
| **Cosine (TF-IDF)** | Wortlaut-Ähnlichkeit über Schlüsselwörter | TF-IDF-Vektoren → Kosinus | 0 – 1 | < 0.60 = schwach · > 0.75 = gut | Terminologie prüfen |
| **SBERT Cosine** | Semantische Nähe trotz Paraphrasen | all-MiniLM Embeddings → Cosine | 0 – 1 | < 0.75 = schwach · > 0.85 = gut | Sinnlücken schließen |
| **BERTScore F1** | Bedeutungs-Overlap auf Tokenebene | mBERT → Precision/Recall → F1 | 0 – 1 | < 0.85 = kritisch · > 0.90 = sehr gut | fehlende Details ergänzen |
| **ROUGE-L F1** | Abdeckung inkl. Reihenfolge | Longest Common Subsequence | 0 – 1 | < 0.45 = niedrig · > 0.60 = gut | Reihenfolge prüfen |
| **ROUGE-1 / 2 P/R/F1** | n-Gram-Overlap (1-/2-Gramme) | Wort- / Wortpaar-Overlap | 0 – 1 | Recall≫Precision → zu lang | Länge anpassen |
| **BLEU-1..4** | Wortlaut-Treue (klassisch) | Geometrisches Mittel + Brevity Penalty | 0 – 1 | < 0.15 = niedrig · > 0.25 = gut | Paraphrasen prüfen |

---

## 2️⃣ Zeit- & Alignment-Metriken

| **Kennzahl** | **Beschreibung** | **Formel (vereinfacht)** | **Skala** | **Interpretation** | **Empfehlung** |
|:--------------|:-----------------|:--------------------------|:-----------|:-------------------|:---------------|
| **Timing Accuracy** | Zeitliche Präzision Ref ↔ Hyp | 1 − (Δ-Mittel / 5 s) → [0–1] | 0 – 1 | ≥ 0.85 = gut · < 0.70 = kritisch | Zeitmarken prüfen |
| **time_overlap** | Überlappung der Zeitintervalle | \|A∩B\| / \|A∪B\| | 0 – 1 | > 0.80 = gut · < 0.60 = gering | Alignment justieren |
| **Δ-Start / Δ-Ende / Δ-Mittel (s)** | Start- / End- / Durchschnitts-Abweichung | Absolutwert in Sek. | – | < 2 = gut · > 5 = hoch | Einsätze synchronisieren |
| **Ref- / Hyp-Index** | Zuordnung der Segmente | TF-IDF + Zeitgewichtung | Integer | – | Plausibilität prüfen |

---

## 3️⃣ Längen- & Silben-Maße

| **Kennzahl** | **Beschreibung** | **Berechnung** | **Interpretation** | **Empfehlung** |
|:--------------|:-----------------|:----------------|:-------------------|:---------------|
| **Wörter Ref/Hyp** | Segmentlänge in Tokens | Token-Zählung | Hyp≪Ref → zu kurz | ergänzen / kürzen |
| **len_ratio (H/R)** | Verhältnis Wortanzahl | Wörter_Hyp / Wörter_Ref | 0.85–1.15 = ideal | Länge angleichen |
| **Silben Ref/Hyp** | Indikator für Sprechdauer | heuristische Zählung | Hyp≫Ref → langsamer | Tempo anpassen |
| **syl_ratio (H/R)** | Verhältnis Silbenanzahl | Silben_Hyp / Silben_Ref | 0.85–1.15 = ideal | Lesetempo justieren |
| **Flesch–Amstad (FRE)** | Lesbarkeits-Index DE | 180 − ASL − (58.5 × ASW) | > 60 = leicht · < 40 = schwer | Syntax vereinfachen |

---

## 4️⃣ Heuristische Merkmale

| **Kennzahl** | **Bedeutung** | **Berechnung** | **Interpretation** | **Empfehlung** |
|:--------------|:--------------|:----------------|:-------------------|:----------------|
| **Farbdetail Ref/Hyp** | visuelle Präzision | Lemma ∈ Farbvokabular | Hyp≪Ref → Farben fehlen | Farben benennen |
| **Bewegung Ref/Hyp** | Dynamik / Handlung | Lemma ∈ Verb-Liste | Hyp≪Ref → Bewegung fehlt | Bewegungsverben ergänzen |

---

## 5️⃣ RAGAS-ähnliche Metriken (0 – 1)

| **Kennzahl** | **Idee** | **Ampel (Gelb/Grün)** | **Interpretation** | **Maßnahme bei niedrig** |
|:--------------|:----------|:----------------------|:-------------------|:--------------------------|
| **semantic_similarity** | Bedeutungsnähe Ref↔Hyp | 0.70 / 0.85 | Inhalt deckungsgleich? | Kernaussagen angleichen |
| **answer_similarity** | Formulierungsnähe | 0.70 / 0.85 | Ausdruck ähnlich? | Terminologie vereinheitlichen |
| **faithfulness** | Faktentreue | 0.60 / 0.75 | keine Halluzinationen | falsche Details entfernen |
| **answer_relevancy** | thematische Relevanz | 0.60 / 0.75 | Fokus korrekt? | Nebenaspekte streichen |
| **coverage** | Abdeckungsgrad Ref | 0.50 / 0.70 | Vollständigkeit? | fehlende Inhalte ergänzen |
| **conciseness** | Prägnanz | 0.60 / 0.80 | kompakt / redundanzfrei? | Füllwörter streichen |
| **fluency (FRE)** | Sprachfluss / Lesbarkeit | 0.60 / 0.75 | flüssig formuliert? | Satzstruktur vereinfachen |

---

## 6️⃣ MDR-Rubrics (0 – 100 Punkte)

| **Rubrik** | **Zielgröße** | **Herleitung (vereinfacht)** | **Bedeutung** |
|:------------|:--------------|:------------------------------|:---------------|
| **Stil / Konzision (Wörter)** | Textlänge | (1 − \|1 − len_ratio\|) × 100 | Umfang vergleichbar |
| **Stil / Konzision (Silben)** | Tempo / Sprechdauer | (1 − \|1 − syl_ratio\|) × 100 | gleiches Tempo |
| **Lesehärte (FRE)** | Verständlichkeit Hyp | normierter FRE × 100 | höhere = leichter |
| **Visuelle Klarheit** | Farbinfos vollständig | (1 − \|Farb-Gap\|) × 100 | visuelle Präzision |
| **Handlungsführung** | Bewegungsverben vollständig | (1 − \|Bewegungs-Gap\|) × 100 | Dynamik |
| **Inhaltsdeckung / Kohärenz** | Inhalt + Struktur | Mittel (ROUGE-L F1 + SBERT) | semantische Konsistenz |
| **Timing-Genauigkeit** | Synchronität | Timing Accuracy × 100 | zeitliche Präzision |

> **Gesamtnote:** gewichtetes Mittel aller Rubrics (standardmäßig gleich gewichtet)

---

## 7️⃣ Streuung & Stabilität (CV %)

| **Kennzahl** | **Beschreibung** | **Formel** | **Richtwerte** | **Interpretation** |
|:--------------|:-----------------|:------------|:----------------|:-------------------|
| **CV %** | Variationskoeffizient pro Metrik | 100 × (Std / Mittelwert) | ≤ 20 % = stabil · ≤ 40 % = moderat · > 40 % = volatil | hohe CV → Qualität schwankt |

---

## 8️⃣ Standard-Ampelschwellen (`THRESHOLDS`)

| **Kennzahl** | **Gelb** | **Grün** |
|:--------------|:----------|:----------|
| Cosine (TF-IDF) | 0.60 | 0.75 |
| SBERT Cosine | 0.75 | 0.85 |
| BERTScore F1 | 0.85 | 0.90 |
| ROUGE-L F1 | 0.45 | 0.60 |
| BLEU-4 | 0.15 | 0.25 |
| len_ratio / syl_ratio | 0.85 – 1.15 | – |
| Timing Accuracy | 0.70 | 0.85 |
| time_overlap | 0.60 | 0.80 |
| semantic_similarity | 0.70 | 0.85 |
| answer_similarity | 0.70 | 0.85 |
| faithfulness | 0.60 | 0.75 |
| answer_relevancy | 0.60 | 0.75 |
| coverage | 0.50 | 0.70 |
| conciseness | 0.60 | 0.80 |
| fluency (FRE) | 0.60 | 0.75 |

> Diese Schwellen sind heuristisch und können im UI oder Code an Domäne, Sprechtakt und Stilvorgaben angepasst werden.


In [2]:
# AD-Qualitätsnotebook Pro

# Notwendige Importe von Bibliotheken
import re, math, os
from collections import Counter
from typing import List, Tuple, Dict, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown, Javascript
import warnings, logging
import ipywidgets as W
from datetime import datetime
import time, ipywidgets as widgets
from html import escape
from matplotlib.ticker import ScalarFormatter
from matplotlib.ticker import FormatStrFormatter

# Integriete Fehlerüberpürfung
import traceback

DEBUG = True

def _dbg_here(tag: str, extra: dict | None = None):
    if not DEBUG: 
        return
    try:
        e = extra or {}
        print(f"[DEBUG] {tag} :: " + " | ".join(f"{k}={v}" for k,v in e.items()))
    except Exception:
        pass

def _catch_and_show(stage: str, e: Exception):
    tb = "".join(traceback.format_exception(type(e), e, e.__traceback__))
    print(tb)  # ← so taucht der volle Trace in der Zelle/Konsole auf
    msg.value = (f"<div style='color:#b00020'><b>Fehler in der Berechnung</b> "
                 f"(Stufe <code>{stage}</code>): {escape(str(e))}"
                 f"<pre style='white-space:pre-wrap'>{escape(tb)}</pre></div>")

# --- Warnings & Logging ruhigstellen ---
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*encoder_attention_mask.*")
for name in ["transformers", "bertscore", "sentence_transformers", "torch"]:
    logging.getLogger(name).setLevel(logging.ERROR)

display(HTML("""
<style>

/* HTML-Widget-Wrapper*/
.jupyter-widgets .widget-html,
.jupyter-widgets .widget-html-content{
  overflow: visible !important;
}

/* (Optik) Tab-Titel*/
.widget-tab :is(.p-TabBar-tab, .lm-TabBar-tab){
  margin-right: 6px; padding: 6px 12px;
  border: 1px solid #d7dbe0; border-bottom: none;
  border-radius: 10px 10px 0 0; background:#f7f9fc; color:#222;
}
.widget-tab :is(.p-TabBar-tab.p-mod-current, .lm-TabBar-tab.p-mod-current){
  font-weight: 600; box-shadow: 0 -1px 0 0 #9aa4b2 inset;
}
</style>
"""))

def sticky_styler(styler_obj) -> str:
    try:
        html = styler_obj.to_html()
    except TypeError:
        html = styler_obj.to_html()
    # Einheitliche Hülle: erzeugt horizontales Scrollen bei Bedarf
    return (
        '<div class="ad-xscroll">'
        '  <div class="ad-styler-wrap" style="overflow:visible;max-width:100%">'
        f'{html}'
        '  </div>'
        '</div>'
    )

# ===================================
# ⚙️ Konfiguration & Umgebungs-Check
# ===================================
CONFIG = {
    "LANG": "de",
    "USE_SPACY": True,
    "USE_SBERT": True,
    "USE_BERTSCORE": True,
    "USE_RAGAS": True,          # versucht echte ragas-Metriken; sonst Fallback
    "SYLLABLE_METHOD": "heuristic_de"
}

# Default-Flags (werden im Panel gesetzt)
__COMPUTE_SBERT__     = True
__COMPUTE_BERTSCORE__ = True
__SHOW_RAGAS__        = True
__HAVE_DATA__ = False
# Reentrancy-Guard (verhindert Doppel-Starts)
__RUNNING__ = False

# Ampel-Schwellen (anpassbar)
THRESHOLDS = {
    # Kern
    "Cosine (TF-IDF)": (0.60, 0.75),
    "SBERT Cosine":    (0.75, 0.85),
    "BERTScore F1":    (0.85, 0.90),
    "ROUGE-L F1":      (0.45, 0.60),
    "BLEU-4":          (0.15, 0.25),

    # ROUGE-Details
    "ROUGE-L P": (0.45, 0.60), "ROUGE-L R": (0.45, 0.60),
    "ROUGE-2 P": (0.20, 0.35), "ROUGE-2 R": (0.20, 0.35),
    "ROUGE-1 P": (0.40, 0.60), "ROUGE-1 R": (0.40, 0.60),

    # RAGAS (0..1)
    "RAGAS: semantic_similarity": (0.70, 0.85),
    "RAGAS: answer_similarity":   (0.70, 0.85),
    "RAGAS: faithfulness":        (0.60, 0.75),
    "RAGAS: answer_relevancy":    (0.60, 0.75),
    "RAGAS: coverage":            (0.50, 0.70),
    "RAGAS: conciseness":         (0.60, 0.80),
    "RAGAS: fluency(FRE)":        (0.60, 0.75),

    # Timing
    "Timing Accuracy": (0.70, 0.85),   # grün ab 0.85, gelb ab 0.70
    "time_overlap":    (0.60, 0.80),   # grün ab 0.80
}

# --- Anzeige-Mapping für RAGAS-Header (nur UI, Daten bleiben gleich) ---
RAGAS_DISPLAY_MAP = {
    "RAGAS: semantic_similarity": "RAGAS\nsemantic\nsimilarity",
    "RAGAS: answer_similarity":   "RAGAS\nanswer\nsimilarity",
    "RAGAS: faithfulness":        "RAGAS\nfaithfulness",
    "RAGAS: answer_relevancy":    "RAGAS\nanswer\nrelevancy",
    "RAGAS: coverage":            "RAGAS\ncoverage",
    "RAGAS: conciseness":         "RAGAS\nconciseness",
    "RAGAS: fluency(FRE)":        "RAGAS\nfluency\n(FRE)",
}
DISPLAY_TO_BASE = {v: k for k, v in RAGAS_DISPLAY_MAP.items()}

def _norm_col_key(k: str) -> str:
    if k is None:
        return ""
    k = str(k)
    # Dashes & Whitespace
    k = k.replace("\u2011","-").replace("\u2013","-").replace("\u2014","-")
    k = re.sub(r"\s+", " ", k).strip()
    # Wichtig: "RAGAS:" zu "RAGAS"
    k = k.replace("RAGAS:", "RAGAS")
    # Anzeige → Basis (für Header-Mapping)
    return DISPLAY_TO_BASE.get(k, k)

# Farben
COLOR_OK   = "#d9f2d9"
COLOR_WARN = "#fff2b3"
COLOR_BAD  = "#ffd6d6"

try:    LEN_RATIO_BAND
except: LEN_RATIO_BAND = (0.85, 1.15)
try:    SYL_RATIO_BAND
except: SYL_RATIO_BAND = (0.85, 1.15)

def _color_for_kpi(col, v):
    """Ampelfarbe für KPI; robust gegen Anzeigeheader & Strings."""
    try:
        v = float(v)
    except Exception:
        return ""
    if pd.isna(v):
        return ""

    base_col = _norm_col_key(col)

    # Deltas bei Zeiten
    if base_col in ("Δ-Mittel (s)", "Δ-Start (s)", "Δ-Ende (s)"):
        if v <= 1.0:  return COLOR_OK
        if v <= 3.0:  return COLOR_WARN
        return COLOR_BAD

    # dynamische Bänder
    if base_col == "len_ratio":
        low, high = LEN_RATIO_BAND
        if low <= v <= high: return COLOR_OK
        if (low - 0.15) <= v <= (high + 0.15): return COLOR_WARN
        return COLOR_BAD

    if base_col == "syl_ratio":
        low, high = SYL_RATIO_BAND
        if low <= v <= high: return COLOR_OK
        if (low - 0.15) <= v <= (high + 0.15): return COLOR_WARN
        return COLOR_BAD

    # regulär
    thr = THRESHOLDS.get(base_col)
    if thr is None:
        # Fallback für 0..1-Scores (wenn kein THRESHOLD hinterlegt)
        if 0.0 <= v <= 1.0:
            if v >= 0.75: return COLOR_OK
            if v >= 0.50: return COLOR_WARN
            return COLOR_BAD
        return ""

    y, g = thr
    if v >= g: return COLOR_OK
    if v >= y: return COLOR_WARN
    return COLOR_BAD

def _scroll_top():
    """Scrollt die Notebook-Ansicht sanft nach oben."""
    try:
        display(Javascript(
            "window.scrollTo({top: 0, behavior: 'smooth'});"
        ))
    except Exception:
        # Fallback (sollte praktisch nie gebraucht werden)
        display(HTML("<script>window.scrollTo({top:0,behavior:'smooth'});</script>"))

def _plain_axes(ax):
    """Deaktiviert Scientific-Notation und Offsets für x & y."""
    for axis in ('x', 'y'):
        fmt = ScalarFormatter(useOffset=False)
        fmt.set_scientific(False)
        (ax.xaxis if axis == 'x' else ax.yaxis).set_major_formatter(fmt)

# =======================
# 📂 MDR & KI Loader
# =======================

# (A) robuste Zeitparser
def _parse_time_to_seconds(t: str):
    if t is None: 
        return np.nan
    ts = str(t).strip().replace("\u2009","").replace("\u00a0","")

    # hh:mm:ss(.ms)  |  mm:ss(.ms)
    m = re.match(r"^(?:(\d{1,2}):)?(\d{1,2}):(\d{1,2})(?:[.,](\d+))?$", ts)
    if m:
        hh = int(m.group(1) or 0)
        mm = int(m.group(2))
        ss = int(m.group(3))
        ms = m.group(4)
        frac = float(f"0.{ms}") if ms else 0.0
        return hh*3600 + mm*60 + ss + frac

    # reine Sekunden (ggf. mit Dezimalen)
    try:
        return float(ts)
    except Exception:
        return np.nan

def _fmt_seconds(sec: float):
    if pd.isna(sec): return ""
    sec=int(round(float(sec)))
    if sec<3600: m,s=divmod(sec,60); return f"{m:02d}:{s:02d}"
    h,r=divmod(sec,3600); m,s=divmod(r,60); return f"{h:d}:{m:02d}:{s:02d}"

# (B) Loader für TXT: 3 Spalten (start\tende\ttext) ODER 2 Spalten (time\ttext)
def load_ad_txt(path: str) -> pd.DataFrame:
    df = None
    # 1) Versuche TSV
    for header in [0, None]:
        try:
            tmp = pd.read_csv(path, sep="\t", header=header, dtype=str, engine="python")
            if tmp.shape[1] >= 3:
                tmp = tmp.iloc[:, :3]
                tmp.columns = ["start", "ende", "text"]
                df = tmp
                break
            if tmp.shape[1] == 2:
                tmp.columns = ["start", "text"]
                tmp["ende"] = ""                     # <— WICHTIG
                tmp = tmp[["start", "ende", "text"]] # konsistente Reihenfolge
                df = tmp
                break
        except Exception:
            pass

    # 2) Fallback: freies Parsen (1–3 Felder pro Zeile)
    if df is None:
        rows = []
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.rstrip("\n")
                parts = line.split("\t") if "\t" in line else re.split(r"\s{2,}", line)
                parts = [p.strip() for p in parts if p.strip()]
                if len(parts) >= 3:
                    rows.append(parts[:3])
                elif len(parts) == 2:
                    rows.append([parts[0], "", parts[1]])
                elif len(parts) == 1:
                    rows.append(["", "", parts[0]])
        df = pd.DataFrame(rows, columns=["start", "ende", "text"])

    # 3) Säubern (nur vorhandene Spalten)
    for c in [c for c in ["start", "ende", "text"] if c in df.columns]:
        df[c] = (
            df[c].fillna("")
                 .astype(str)
                 .str.replace(r"\s+", " ", regex=True)
                 .str.strip()
        )

    # 4) Zeiten berechnen
    df["t_start"] = df["start"].map(_parse_time_to_seconds)
    has_end = df["ende"].replace("", np.nan).notna().any()
    df["t_end"]   = df["ende"].map(_parse_time_to_seconds) if has_end else np.nan

    # Falls nur Startzeiten: Ende = nächster Start bzw. +2s
    # korrekt: nur dort füllen, wo Ende fehlt oder vor dem Start liegt
    mask_has_start = df["t_start"].notna()
    # Ende nur auffüllen, wenn NaN/leer oder kleiner als Start (defekt)
    mask_need_end = df["t_end"].isna() | (df["t_end"] < df["t_start"])
    
    # Liste der Startzeiten für die "nächster Start"-Logik
    starts = df.loc[mask_has_start, "t_start"].to_numpy()
    fill_vals = [starts[i+1] if i < len(starts)-1 else starts[i] + 2.0 for i in range(len(starts))]
    
    # nur bei Zeilen anwenden, die ein Ende brauchen
    idxs = df.index[mask_has_start & mask_need_end]
    df.loc[idxs, "t_end"] = [fill_vals[i] for i in range(len(idxs))]

    # formatierte Zeiten
    df["start"] = df["t_start"].map(_fmt_seconds)
    df["ende"]  = df["t_end"].map(_fmt_seconds)
    return df[["start", "ende", "text", "t_start", "t_end"]]

# ==========================================================
# UI – Upload + Buttons (Laden / Berechnen / Reset / Format)
# ==========================================================

# --- Breiten/Layouts
BTN_W   = "220px"
DESC_W  = "92px"
PROG_W  = "100%"
UPLOAD_W = BTN_W

u_mdr = W.FileUpload(accept=".txt", multiple=False, layout=W.Layout(width=UPLOAD_W))
u_ki  = W.FileUpload(accept=".txt", multiple=False, layout=W.Layout(width=UPLOAD_W))

btn_load  = W.Button(description="Laden",     icon="upload", button_style="primary",
                     layout=W.Layout(width=BTN_W, height="36px"))
btn_calc  = W.Button(description="Berechnen", icon="gear",   button_style="primary",
                     layout=W.Layout(width=BTN_W, height="36px"))
btn_reset = W.Button(description="Reset",     icon="trash",  button_style="",
                     layout=W.Layout(width="140px", height="36px"))

# Fortschritt
p_load = W.IntProgress(description="Laden:", min=0, max=100, value=0, bar_style="info",
                       layout=W.Layout(width=PROG_W), style={"description_width": DESC_W})
p_calc = W.IntProgress(description="Berechnen:", min=0, max=100, value=0, bar_style="info",
                       layout=W.Layout(width=PROG_W), style={"description_width": DESC_W})

# >>> Overflow-Settings für Widgets ...
for _w in (u_mdr, u_ki, btn_load, btn_calc, btn_reset, p_load, p_calc):
    _w.layout.overflow = 'visible'

def _progress_calc(frac: float):
    """Setzt den Berechnungsfortschritt (0..1 → 0..100)."""
    try:
        frac = float(frac)
    except Exception:
        frac = 0.0
    p_calc.value = max(0, min(100, int(round(frac * 100))))
    p_calc.bar_style = "success" if p_calc.value >= 100 else "info"

progress_card = W.VBox(
    [W.HTML("<div style='font-weight:600;margin-bottom:6px'>Fortschrittsanzeige</div>"), p_load, p_calc],
    layout=W.Layout(padding="10px 12px", border="1px solid #e5e7eb", border_radius="8px",
                    margin="6px 0 0 0", width="100%")
)

msg = W.HTML()
btn_calc.disabled = True

def _save_upload(widget: W.FileUpload, target_path: str) -> str | None:
    v = widget.value
    if not v: return None
    meta = next(iter(v.values())) if isinstance(v, dict) else v[0]
    content = meta["content"] if isinstance(meta, dict) else getattr(meta, "content", None)
    if content is None: raise ValueError("Upload-Widget ohne 'content'")
    with open(target_path, "wb") as f: f.write(content)
    try: widget.value = ()
    except Exception:
        try: widget.value = {}
        except Exception: pass
    return target_path

def _set_compute_enabled():
    have = isinstance(globals().get("ref_segs"), list) and len(globals().get("ref_segs", [])) \
        and isinstance(globals().get("hyp_segs"), list) and len(globals().get("hyp_segs", []))
    btn_calc.disabled = not bool(have)

def _unique_path(base: str, ext: str) -> str:
    ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    path = f"/mnt/data/{base}_{ts}{ext}"
    if not os.path.exists(path):
        return path
    i = 1
    while True:
        alt = f"/mnt/data/{base}_{ts}_{i}{ext}"
        if not os.path.exists(alt):
            return alt
        i += 1

def _export_txt_safely(df: pd.DataFrame) -> tuple[str, str | None]:
    """Speichert 1) UTF-8-SIG (Excel freundlich) und optional 2) UTF-16-TSV."""
    if df is None or df.empty:
        raise RuntimeError("Keine Ergebnisse – bitte zuerst Berechnen.")
    out_txt = _unique_path("ad_compare_export", ".txt")
    # numerische Spalten runden (nur floats)
    num_cols = [c for c in df.columns if pd.api.types.is_float_dtype(df[c])]
    if num_cols:
        df[num_cols] = df[num_cols].round(4)
    # 1) UTF-8 mit BOM → Excel liest sauber
    df.to_csv(out_txt, sep="\t", index=False, encoding="utf-8-sig")

    # 2) optional: UTF-16 (manche Excel-Builds mögen das)
    out_tsv16 = None
    try:
        out_tsv16 = _unique_path("ad_compare_export_excel", ".tsv")
        df.to_csv(out_tsv16, sep="\t", index=False, encoding="utf-16", encoding_errors="ignore")
    except Exception:
        out_tsv16 = None
    return out_txt, out_tsv16

# ---- Tabs-Output ----
tabs_out = W.Output()

charts_out = W.Output()
summary_out = W.Output()

charts_out.layout.margin = '14px 0 0 0'

with charts_out:
    charts_out.clear_output()
    display(HTML("<em>Noch keine Charts – bitte zuerst <b>Berechnen</b>.</em>"))
with summary_out:
    summary_out.clear_output()
    display(HTML("<em>Noch keine Kennzahlen – bitte zuerst <b>Berechnen</b>.</em>"))
    
tabs_out.layout.max_height = 'none'
tabs_out.layout.overflow_y = 'visible'
tabs_out.layout.overflow_x = 'hidden'

with tabs_out:
    tabs_out.clear_output()
    display(HTML("<em>Noch keine Daten. Bitte MDR & KI laden und dann auf <b>Berechnen</b> klicken.</em>"))

def _reset_state():
    for k in ["ref_segs","hyp_segs","pairs","df","table","ragas_seg","ref_tok","hyp_tok","S","report"]:
        globals().pop(k, None)
    globals()["__HAVE_DATA__"] = False
    p_load.value = 0; p_load.bar_style = "info"
    p_calc.value = 0; p_calc.bar_style = "info"
    msg.value = "<div style='color:#4b5563'>Zurückgesetzt. Bitte neue Dateien laden.</div>"
    _set_compute_enabled()
    with tabs_out:
        tabs_out.clear_output()
        display(HTML("<em>Noch keine Daten. Bitte MDR & KI laden und dann auf <b>Berechnen</b> klicken.</em>"))
    with charts_out:
        charts_out.clear_output()
        display(HTML("<em>Noch keine Charts – bitte zuerst <b>Berechnen</b>.</em>"))

header = W.HTML(
    '<div style="padding:10px 12px;border:1px solid #e5e7eb;border-radius:12px;background:#fafafa">'
    '<div style="font-weight:600;margin-bottom:6px">Daten laden (MDR & KI)</div>'
    '<ol style="color:#4b5563;font-size:12px;margin:0 0 4px 18px;padding:0;line-height:1.4">'
    '<li>Beide <b>.txt</b>-Dateien über die Upload-Buttons auswählen.</li>'
    '<li>Auf <b>Laden</b> klicken.</li>'
    '<li>Mit <b>Berechnen</b> die Auswertung starten.</li>'
    '<li><b>Reset</b> setzt alles zurück für einen neuen Vergleich.</li>'
    '</ol>'
    '</div>'
)

# Upload-Spalten + Grid
col_mdr = W.VBox([W.HTML("<b>MDR .txt</b>"), u_mdr], layout=W.Layout(width=BTN_W))
col_ki  = W.VBox([W.HTML("<b>KI .txt</b>"),  u_ki ], layout=W.Layout(width=BTN_W))
spacer_top = W.Box(layout=W.Layout(width=BTN_W, height="32px"))

grid_panel = W.GridBox(
    children=[col_mdr, col_ki, spacer_top, btn_load, btn_calc, btn_reset],
    layout=W.Layout(
        grid_template_columns=f"{BTN_W} {BTN_W} {BTN_W}",
        grid_auto_rows="min-content",
        grid_gap="10px",
        align_items="center",
        justify_items="flex-start",  # <— statt "start"
    )
)

# >>> NEU: Container sollen selbst nicht horizontal scrollen
col_mdr.layout.overflow = 'visible'
col_ki.layout.overflow  = 'visible'
grid_panel.layout.overflow = 'visible'
progress_card.layout.overflow = 'visible'

# Obercontainer
box = W.VBox(
    [header, grid_panel, progress_card, tabs_out, msg],
    layout=W.Layout(gap="10px", width="100%", overflow='visible', overflow_x='hidden')
)

display(box)

# --- FIX: nur Zeilen mit Text behalten (führt auch die Zeitspalten mit) ---
def _filter_nonempty_text(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty: 
        return df
    m = df["text"].astype(str).str.strip().ne("")
    return df.loc[m].reset_index(drop=True)

def on_load_click(_):
    globals()["__RUNNING__"] = False

    try:
        msg.value = ""
        p_load.value = 5;  p_load.bar_style = "info"
        for k in ["pairs","df","table","ragas_seg","ref_tok","hyp_tok","S","report"]:
            globals().pop(k, None)
        globals()["__HAVE_DATA__"] = False
        _set_compute_enabled()

        mdr_saved = _save_upload(u_mdr, "/mnt/data/_mdr.txt")
        ki_saved  = _save_upload(u_ki,  "/mnt/data/_ki.txt")
        if not mdr_saved or not ki_saved:
            msg.value = "<div style='color:#b00020'>Bitte beide Dateien auswählen und erneut auf <b>Laden</b> klicken.</div>"
            p_load.value = 0; p_load.bar_style = "danger"
            return

        p_load.value = 30
        mdr_df = load_ad_txt(mdr_saved);  p_load.value = 55
        ki_df  = load_ad_txt(ki_saved);   p_load.value = 75

        # Sekundenfelder ergänzen
        for _df in (mdr_df, ki_df):
            _df["start_sec"] = _df["start"].map(_parse_time_to_seconds)
            _df["ende_sec"]  = _df["ende"].map(_parse_time_to_seconds)

        # --- FIX: konsequent nur Zeilen mit Text behalten ---
        mdr_df = _filter_nonempty_text(mdr_df)
        ki_df  = _filter_nonempty_text(ki_df)

        # für spätere Exporte/Zeitspalten aufbewahren (bereits gefiltert!)
        globals()["mdr_df"] = mdr_df.copy()
        globals()["ki_df"]  = ki_df.copy()

        # >>> PATCH START: Segmente exakt wie geladen (auch leere Pausen behalten)
        globals()["ref_segs"] = mdr_df["text"].astype(str).tolist()
        globals()["hyp_segs"] = ki_df["text"].astype(str).tolist()
        # >>> PATCH END

        ##################################################################################
        # Debug, wenn Variablen existieren
        _dbg_here("after_load", {
            "mdr_rows": len(mdr_df),
            "ki_rows": len(ki_df),
            "mdr_nan_starts": int(pd.isna(mdr_df["start_sec"]).sum()),
            "ki_nan_starts": int(pd.isna(ki_df["start_sec"]).sum()),
        })
        ##################################################################################
        
        p_load.value = 100; p_load.bar_style = "success"
        msg.value = (f"<div style='color:#2e7d32'>OK – {len(ref_segs)} Referenz- und {len(hyp_segs)} KI-Segmente geladen.</div>")
        _scroll_top()
    except Exception as e:
        p_load.bar_style = "danger"
        msg.value = f"<div style='color:#b00020'>Fehler beim Laden: {e}</div>"
    finally:
        _set_compute_enabled()

btn_load.on_click(on_load_click)
btn_calc.on_click(lambda _: compute_pipeline())
btn_reset.on_click(lambda _: _reset_state())

# ==================================
# 🔌 Optionale Bibliotheken - RAGAS
# ==================================
_HAS_ST = CONFIG["USE_SBERT"]
_HAS_BERTSCORE = CONFIG["USE_BERTSCORE"]
_HAS_SPACY = CONFIG["USE_SPACY"]
_HAS_RAGAS = CONFIG["USE_RAGAS"]

try:
    from sentence_transformers import SentenceTransformer
except Exception:
    _HAS_ST = False

try:
    from bert_score import score as bertscore_score
except Exception:
    _HAS_BERTSCORE = False

_nlp_de = None
if _HAS_SPACY:
    try:
        import spacy
        try:
            _nlp_de = spacy.load("de_core_news_sm", disable=["ner","parser","textcat"])
        except Exception:
            _HAS_SPACY = False
    except Exception:
        _HAS_SPACY = False

# RAGAS (optional, defensiv)
_ragas_available = False
_ragas_metrics = {}
if _HAS_RAGAS:
    try:
        import ragas  # noqa
        try:
            from ragas.metrics import (
                faithfulness as rg_faithfulness,
                answer_relevancy as rg_answer_relevancy,
            )
            _ragas_metrics["faithfulness"] = rg_faithfulness
            _ragas_metrics["answer_relevancy"] = rg_answer_relevancy
        except Exception:
            pass
        try:
            from ragas.metrics import semantic_similarity as rg_semantic_similarity
            _ragas_metrics["semantic_similarity"] = rg_semantic_similarity
        except Exception:
            pass
        try:
            from ragas.metrics import answer_similarity as rg_answer_similarity
            _ragas_metrics["answer_similarity"] = rg_answer_similarity
        except Exception:
            pass
        try:
            from ragas.metrics import coverage as rg_coverage
            _ragas_metrics["coverage"] = rg_coverage
        except Exception:
            pass
        try:
            from ragas.metrics import conciseness as rg_conciseness
            _ragas_metrics["conciseness"] = rg_conciseness
        except Exception:
            pass
        _ragas_available = len(_ragas_metrics) > 0
    except Exception:
        _ragas_available = False

# ====================================================
# 🎨 Domänenspezifische Erkennung (Farben & Bewegung)
# ====================================================
COLOR_STEMS = [
    "schwarz","weiß","weiss","grau","rot","blau","grün","gruen","gelb","orange",
    "violett","lila","braun","beige","gold","silber","türkis","tuerkis","magenta","rosa"
]
COLOR_PREFIXES = ["hell","dunkel","knall","neon","pastell"]

def _strip_color_affixes(piece: str) -> str:
    x = piece
    x = re.sub(r"farb\w+$", "", x)
    for pref in COLOR_PREFIXES:
        if x.startswith(pref):
            x = x[len(pref):]
            break
    return x

def has_color_tokens(tokens: list[str]) -> bool:
    for tok in (t.lower() for t in tokens):
        for part in tok.split('-'):
            base = _strip_color_affixes(part)
            for stem in COLOR_STEMS:
                if base.startswith(stem):
                    if stem == "rosa" and not re.match(r"rosa($|[a-zäöüß]*?(?:e|en|em|er|es|farb))", base):
                        continue
                    return True
    return False

COLOR_LEMMAS = set(COLOR_STEMS)
MOTION_LEMMAS = {
    "gehen","kommen","laufen","rennen","springen","schreiten",
    "drehen","heben","senken","zeigen","blicken","schauen",
    "fahren","wenden","betreten","treten","setzen","nähern","bewegen"
}
MOTION_MULTI = {"stehen bleiben","sich setzen","sich drehen","sich nähern","aufstehen","hinsetzen"}

def word_tokenize(text: str) -> List[str]:
    text = re.sub(r"\s+"," ", text.lower()).strip()
    return re.findall(r"[A-Za-zÄÖÜäöüß0-9\-]+", text)

def lemmas_de(text: str) -> list[str]:
    if _HAS_SPACY and _nlp_de is not None:
        doc = _nlp_de(text)
        return [t.lemma_.lower() for t in doc if not t.is_punct and not t.is_space]
    return [t.lower() for t in word_tokenize(text)]

def has_motion_text(text: str) -> bool:
    L = lemmas_de(text)
    lemma_str = " ".join(L)
    if any(p in lemma_str for p in MOTION_MULTI): return True
    return any(l in MOTION_LEMMAS for l in L)

def has_color_lemma(tokens: list[str]) -> bool:
    if not _HAS_SPACY: return has_color_tokens(tokens)
    doc = _nlp_de(" ".join(tokens))
    lemmas = {t.lemma_.lower() for t in doc}
    texts  = [t.text.lower() for t in doc]
    if lemmas & COLOR_LEMMAS: return True
    for t in texts:
        for part in t.split("-"):
            base = _strip_color_affixes(part)
            if base in COLOR_LEMMAS: return True
            if any(base.endswith(c) for c in COLOR_LEMMAS): return True
    return False

def has_motion_tokens(tokens: list[str]) -> bool:
    G = {
        "gehen","geht","ging","gegangen","kommen","kommt","kam","gekommen","laufen","läuft","lief","gelaufen","laeuft",
        "rennen","rennt","rannte","gerannt","springen","springt","sprang","gesprungen","drehen","dreht","drehte","gedreht",
        "heben","hebt","hob","gehoben","senken","senkt","senkte","gesenkt","zeigen","zeigt","zeigte","gezeigt",
        "blicken","blickt","blickte","geblickt","schauen","schaut","schaute","geschaut",
        "schreiten","schreitet","schritt","geschritten","fahren","fährt","fuhr","gefahren","faehrt","wenden","wendet","wandte","gewandt"
    }
    toks = {t.lower() for t in tokens}
    return any(t in G for t in toks)

# =========================
# 🔤 Satz- & Wort-Utilities
# =========================
def normalize_text(text: str) -> str:
    return re.sub(r"\s+"," ", text.lower()).strip()

def sent_tokenize(text: str) -> List[str]:
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if len(lines) >= 2 and (sum(len(l) for l in lines)/max(len(lines),1)) > 10:
        return lines
    parts = re.split(r"(?<=[\.\!\?])\s+", text)
    return [p.strip() for p in parts if p.strip()]

def ngrams(tokens: List[str], n: int):
    return [tuple(tokens[i:i+n]) for i in range(0, len(tokens)-n+1)]

# =======================
# 🗣️ BLEU / ROUGE
# =======================
def bleu_score(reference: List[str], hypothesis: List[str], max_n: int = 4) -> float:
    precisions = []
    for n in range(1, max_n+1):
        ref_ngrams = Counter(ngrams(reference, n))
        hyp_ngrams = Counter(ngrams(hypothesis, n))
        total = sum(hyp_ngrams.values())
        if total == 0: precisions.append(0.0); continue
        overlap = sum(min(cnt, ref_ngrams.get(ng,0)) for ng, cnt in hyp_ngrams.items())
        precisions.append(overlap / total if total > 0 else 0.0)
    ref_len, hyp_len = len(reference), len(hypothesis)
    if hyp_len == 0: return 0.0
    bp = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / max(hyp_len, 1))
    if any(p == 0 for p in precisions): return 0.0
    log_prec = sum(math.log(p) for p in precisions) / len(precisions)
    return float(bp * math.exp(log_prec))

def rouge_n(reference: List[str], hypothesis: List[str], n: int = 1) -> Dict[str, float]:
    ref_ngrams = Counter(ngrams(reference, n))
    hyp_ngrams = Counter(ngrams(hypothesis, n))
    overlap = sum(min(cnt, hyp_ngrams.get(ng,0)) for ng, cnt in ref_ngrams.items())
    ref_total = sum(ref_ngrams.values()); hyp_total = sum(hyp_ngrams.values())
    recall = overlap / ref_total if ref_total > 0 else 0.0
    precision = overlap / hyp_total if hyp_total > 0 else 0.0
    f1 = (2*precision*recall)/(precision+recall) if (precision+recall) > 0 else 0.0
    return {"precision": float(precision), "recall": float(recall), "f1": float(f1)}

def lcs_length(a: List[str], b: List[str]) -> int:
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            dp[i][j] = dp[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]

def rouge_l(reference: List[str], hypothesis: List[str]) -> Dict[str, float]:
    lcs = lcs_length(reference, hypothesis)
    ref_len, hyp_len = len(reference), len(hypothesis)
    recall = lcs / ref_len if ref_len > 0 else 0.0
    precision = lcs / hyp_len if hyp_len > 0 else 0.0
    f1 = (2*precision*recall)/(precision+recall) if (precision+recall) > 0 else 0.0
    return {"precision": float(precision), "recall": float(recall), "f1": float(f1)}

# =======================
# 🔎 TF-IDF & Cosine
# =======================
def build_tfidf_vectors(docs_tokens: List[List[str]]):
    vocab = sorted(set(t for doc in docs_tokens for t in doc))
    term_index = {t:i for i,t in enumerate(vocab)}
    tf = np.zeros((len(docs_tokens), len(vocab)), dtype=float)
    for di, toks in enumerate(docs_tokens):
        counts = Counter(toks); total = sum(counts.values()) or 1
        for t, c in counts.items():
            tf[di, term_index[t]] = c / total
    df = np.array([sum(1 for doc in docs_tokens if t in set(doc)) for t in vocab], dtype=float)
    idf = np.log((1 + len(docs_tokens)) / (1 + df)) + 1.0
    tfidf = tf * idf
    return tfidf, vocab

def cosine_similarity_matrix(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-12)
    return A_norm @ B_norm.T

def greedy_alignment(ref_segments_tokens: List[List[str]], hyp_segments_tokens: List[List[str]]):
    all_docs = ref_segments_tokens + hyp_segments_tokens
    tfidf, _ = build_tfidf_vectors(all_docs)
    n_ref = len(ref_segments_tokens)
    A, B = tfidf[:n_ref, :], tfidf[n_ref:, :]
    S = cosine_similarity_matrix(A, B)
    used_hyp = set(); pairs = []
    for r in range(n_ref):
        order = np.argsort(-S[r]); h_choice = None
        for h in order:
            if h not in used_hyp: h_choice = h; break
        if h_choice is None: h_choice = int(order[0])
        pairs.append((r, h_choice, float(S[r, h_choice]))); used_hyp.add(h_choice)
    return pairs, S

# ===== Timing-Hilfen =====
def _sec(x): 
    try: return float(x)
    except: return np.nan

def time_overlap_and_deltas(s1, e1, s2, e2):
    
    s1, e1, s2, e2 = map(_sec, (s1, e1, s2, e2))
    if pd.isna(s1) or pd.isna(s2): 
        return np.nan, np.nan, np.nan, np.nan
        
    # Ende ableiten, falls leer
    if pd.isna(e1) or e1 < s1: e1 = s1
    if pd.isna(e2) or e2 < s2: e2 = s2
        
    # Überlappung
    inter = max(0.0, min(e1, e2) - max(s1, s2))
    union = max(e1, e2) - min(s1, s2) if max(e1, e2) >= min(s1, s2) else 0.0
    if union <= 0:
        # beide Intervalle sind punktförmig
        overlap = 1.0 if abs(s1 - s2) < 1e-6 else 0.0
    else:
        overlap = inter / (union + 1e-9)
        
    # Deltas
    d_start = abs(s1 - s2)
    d_end   = abs(e1 - e2)
    d_mean  = 0.5*(d_start + d_end)
    
    return overlap, d_start, d_end, d_mean

def time_aware_alignment(
    mdr_df: pd.DataFrame,
    ki_df: pd.DataFrame,
    S_text: np.ndarray | None = None,
    *,
    w_time: float = 0.6,
    w_text: float = 0.4,
    time_window_s: float = 45.0,
    tol_full: float = 5.0,
    hard_window: bool = True,
    monotonic: bool = True,
    allow_reuse: bool = False,
    jump_penalty: float = 0.04,
    # NEU:
    allow_unmatched: bool = True,     # Ref darf „kein Partner“ haben
    min_time_sim: float = 0.15,       # Mindest-Zeitähnlichkeit (0..1)
    min_final_score: float = 0.20     # Mindest-Gesamtscore (0..1)
):
    m_s = pd.to_numeric(mdr_df["start_sec"], errors="coerce").to_numpy(dtype=float)
    k_s = pd.to_numeric(ki_df["start_sec"],  errors="coerce").to_numpy(dtype=float)
    m_e = pd.to_numeric(mdr_df["ende_sec"],  errors="coerce").to_numpy(dtype=float)
    k_e = pd.to_numeric(ki_df["ende_sec"],   errors="coerce").to_numpy(dtype=float)

    D = np.abs(m_s[:, None] - k_s[None, :])
    sim_time = np.clip(1.0 - (D / max(tol_full, 1e-9)), 0.0, 1.0)

    if S_text is None or S_text.shape != sim_time.shape:
        S_text = np.zeros_like(sim_time, dtype=float)

    w_sum = float(w_time + w_text) or 1.0
    S = (float(w_time)/w_sum) * sim_time + (float(w_text)/w_sum) * S_text

    used_h = set()
    pairs  = []
    last_h = -1
    R, H = S.shape

    for r in range(R):
        order = np.argsort(-S[r, :])

        def candidates(window_mult: float, allow_used: bool):
            cond = np.ones(H, dtype=bool)
            if monotonic:
                cond &= (np.arange(H) >= max(last_h, -1))
            if hard_window:
                cond &= (np.abs(k_s - m_s[r]) <= (time_window_s * window_mult))
            if not allow_used and not allow_reuse and len(used_h) > 0:
                m = np.ones(H, dtype=bool); m[list(used_h)] = False; cond &= m
            return [int(h) for h in order if cond[h]]

        # gestaffelte Kandidatensuche
        cands = candidates(1.0, False) or candidates(1.0, True) or candidates(2.0, True)
        if not cands:
            saved = hard_window; hard_window = False
            cands = candidates(1.0, True)
            hard_window = saved
        if not cands and order.size:
            cands = [int(order[0])]

        # Sprung-Penalty
        if cands and jump_penalty and last_h >= 0:
            scores_adj = []
            for h in cands:
                delta = max(0, h - last_h - 1)
                scores_adj.append(S[r, h] - jump_penalty * float(delta))
            h_best = cands[int(np.argmax(scores_adj))]
        else:
            h_best = (cands[0] if cands else None)

        # --- Gate: Mindest-Ähnlichkeiten ---
        choose_none = False
        if allow_unmatched:
            if h_best is None:
                choose_none = True
            else:
                if (sim_time[r, h_best] < float(min_time_sim)) or (S[r, h_best] < float(min_final_score)):
                    choose_none = True

        if choose_none:
            pairs.append((r, None, 0.0))           # kein Partner
            # last_h & used_h bleiben unverändert
            continue

        # regulärer Treffer
        pairs.append((r, int(h_best), float(S[r, h_best])))
        last_h = int(h_best)
        if not allow_reuse:
            used_h.add(int(h_best))

    # Hilfsinfo: unverbrauchte KI-Indizes (Waisen)
    orphan_h = [h for h in range(H) if h not in used_h]
    return pairs, S, orphan_h

# ===========================================
# 🔢 Silben & Lesbarkeit (DE, Flesch–Amstad)
# ===========================================
_VOWELS_DE = set("aeiouyäöü")
def count_syllables_de_word(w: str) -> int:
    w = w.lower()
    w = re.sub(r"[^a-zäöüß]", "", w)
    if not w: return 0
    w2 = re.sub(r"ie", "i", w)
    w2 = re.sub(r"aa|ee|oo", lambda m: m.group(0)[0], w2)
    groups = re.findall(r"[aeiouyäöü]+", w2)
    syl = len(groups)
    if w.endswith("e"): syl += 0
    return max(syl, 1)

def count_syllables_text(text: str) -> int:
    toks = word_tokenize(text)
    return sum(count_syllables_de_word(t) for t in toks)

def flesch_amstad(text: str) -> float:
    sentences = sent_tokenize(text)
    n_sent = max(len(sentences), 1)
    toks = word_tokenize(text)
    n_words = max(len(toks), 1)
    n_syl = sum(count_syllables_de_word(t) for t in toks)
    ASL = n_words / n_sent
    ASW = n_syl / n_words
    fre = 180 - ASL - (58.5 * ASW)
    return float(np.clip(fre, -20, 130))

# =================================================
# 📏 Segmentmetriken (inkl. Lemma-Checks & Silben)
# =================================================
def contains_any(tokens: List[str], vocab: set[str]) -> bool:
    return any(t in vocab for t in tokens)

def segment_metrics(ref_tokens: List[str], hyp_tokens: List[str]) -> Dict[str, Any]:
    res = {}
    res["bleu_1"] = bleu_score(ref_tokens, hyp_tokens, max_n=1)
    res["bleu_2"] = bleu_score(ref_tokens, hyp_tokens, max_n=2)
    res["bleu_3"] = bleu_score(ref_tokens, hyp_tokens, max_n=3)
    res["bleu_4"] = bleu_score(ref_tokens, hyp_tokens, max_n=4)
    r1 = rouge_n(ref_tokens, hyp_tokens, n=1)
    r2 = rouge_n(ref_tokens, hyp_tokens, n=2)
    rl = rouge_l(ref_tokens, hyp_tokens)
    for k,v in r1.items(): res[f"rouge1_{k}"] = v
    for k,v in r2.items(): res[f"rouge2_{k}"] = v
    for k,v in rl.items(): res[f"rougeL_{k}"] = v

    # Heuristiken
    res["ref_has_color"]  = has_color_lemma(ref_tokens)
    res["hyp_has_color"]  = has_color_lemma(hyp_tokens)
    ref_txt = " ".join(ref_tokens)
    hyp_txt = " ".join(hyp_tokens)
    res["ref_has_motion"] = has_motion_text(ref_txt) if _HAS_SPACY else has_motion_tokens(ref_tokens)
    res["hyp_has_motion"] = has_motion_text(hyp_txt) if _HAS_SPACY else has_motion_tokens(hyp_tokens)

    # Längen/Silben
    res["len_ref"] = len(ref_tokens); res["len_hyp"] = len(hyp_tokens)
    res["syl_ref"] = sum(count_syllables_de_word(t) for t in ref_tokens)
    res["syl_hyp"] = sum(count_syllables_de_word(t) for t in hyp_tokens)
    return res

# =======================
# SBERT & BERTScore
# =======================
_ST_MODEL = None
def get_st_model(model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
    global _ST_MODEL
    if not _HAS_ST: return None
    if _ST_MODEL is None:
        _ST_MODEL = SentenceTransformer(model_name)
    return _ST_MODEL

def sentence_embeddings(sentences: List[str]):
    if not _HAS_ST: return None
    model = get_st_model()
    try:
        return model.encode(sentences, convert_to_numpy=True, normalize_embeddings=True)
    except Exception:
        return None

def bertscore_pairs(ref_segments, hyp_segments, model_type: str = "bert-base-multilingual-cased"):
    if not _HAS_BERTSCORE: return None
    try:
        f1 = []
        for r, h in zip(ref_segments, hyp_segments):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                P, R, F1 = bertscore_score([h], [r], model_type=model_type, verbose=False)
            f1.append(float(F1.mean().item()))
        return f1
    except Exception:
        return None

# ==================================
# Stopwörter (für Coverage-Fallback)
# ==================================
GER_STOP = {
    "der","die","das","und","oder","ein","eine","einer","einem","einen","den","dem","des",
    "zu","auf","im","in","am","an","ist","war","sind","sein","mit","von","für","als","auch",
    "sich","sie","er","es","wir","ihr","ihn","ihm","man","dass","so","wie","nicht","nur","noch","schon"
}

def content_tokens(tokens: List[str]) -> List[str]:
    return [t for t in tokens if t not in GER_STOP and not t.isdigit()]

# ================================================
# RAGAS (optional) – Fallback-Metriken pro Segment
# ================================================
def ragas_like_metrics(ref_segs: List[str], hyp_segs: List[str], df_base: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for i, (r, h) in enumerate(zip(ref_segs, hyp_segs)):
        rt = word_tokenize(r); ht = word_tokenize(h)
        rL   = df_base.loc[i, "rougeL_f1"]        if "rougeL_f1"        in df_base.columns else np.nan
        rL_P = df_base.loc[i, "rougeL_precision"] if "rougeL_precision" in df_base.columns else np.nan
        rL_R = df_base.loc[i, "rougeL_recall"]    if "rougeL_recall"    in df_base.columns else np.nan
        sbert= df_base.loc[i, "sbert_cosine"]     if "sbert_cosine"     in df_base.columns else np.nan
        bsf1 = df_base.loc[i, "bertscore_f1"]     if "bertscore_f1"     in df_base.columns else np.nan

        def clamp01(x):
            try: return float(np.clip(x, 0, 1))
            except: return np.nan

        sem_sim = sbert if not pd.isna(sbert) else rL
        ans_sim = bsf1  if not pd.isna(bsf1)  else rL
        faith   = rL_R
        ans_rel = rL_P
        ct_r = set(content_tokens(rt)); ct_h = set(content_tokens(ht))
        cov = len(ct_r & ct_h) / (len(ct_r) or 1)
        lr = (len(ht) / (len(rt) or 1))
        conc = 1 - min(1.0, abs(lr - 1.0))
        fre_seg = flesch_amstad(h)
        flu = float(np.clip(fre_seg/100.0, 0, 1))

        rows.append({
            "RAGAS: semantic_similarity": clamp01(sem_sim),
            "RAGAS: answer_similarity":   clamp01(ans_sim),
            "RAGAS: faithfulness":        clamp01(faith),
            "RAGAS: answer_relevancy":    clamp01(ans_rel),
            "RAGAS: coverage":            clamp01(cov),
            "RAGAS: conciseness":         clamp01(conc),
            "RAGAS: fluency(FRE)":        clamp01(flu),
        })
    return pd.DataFrame(rows)     

# --- Reentrancy-Flag oben bleibt ---
__RUNNING__ = False

# >>> PATCH START: sichere Zugriffe
def _safe_tok(tokens_list, idx):
    """Gib Tokenliste für Index zurück; leeres Segment, wenn out-of-range."""
    if 0 <= idx < len(tokens_list):
        return tokens_list[idx]
    return []

def _safe_txt(text_list, idx):
    """Gib Text für Index zurück; leer, wenn out-of-range."""
    if 0 <= idx < len(text_list):
        return text_list[idx]
    return ""
# >>> PATCH END

# =======================
# Pipeline ausführen
# =======================
def compute_pipeline(_=None):
    global df, table, ragas_seg, ref_tok, hyp_tok, pairs, S, __RUNNING__, __HAVE_DATA__

    # 1) Daten vorhanden?
    if not (isinstance(globals().get("ref_segs"), list) and len(ref_segs)
            and isinstance(globals().get("hyp_segs"), list) and len(hyp_segs)):
        msg.value = "<div style='color:#b00020'>Bitte oben beide Dateien laden und dann <b>Berechnen</b> klicken.</div>"
        return

    _scroll_top()

    # 2) Reentrancy
    if __RUNNING__:
        return
    __RUNNING__ = True
    btn_calc.disabled = True

    try:
        _progress_calc(0.05); _dbg_here("start")

        # 3) Tokenisieren & Alignment
        ref_tok = [word_tokenize(s) for s in ref_segs]
        hyp_tok = [word_tokenize(s) for s in hyp_segs]

        # --- NEU (Schritt 9): zuerst Text-Alignment, dann optional Zeit + Text kombinieren ---
        _dbg_here("align_pre") # herausnehmen
        # --- Text-Alignment ---
        pairs_text, S_text = greedy_alignment(ref_tok, hyp_tok)
        
        # --- Zeit + Text kombiniert (robust gegen unterschiedliche Zeilenzahlen) ---
        _mdr = globals().get("mdr_df", None)
        _ki  = globals().get("ki_df",  None)
        USE_TIME_AWARE = True
        
        if USE_TIME_AWARE and (_mdr is not None) and (_ki is not None):
            # S_text ggf. auf (len(_mdr) x len(_ki)) bringen ...
            sim_rows, sim_cols = len(_mdr), len(_ki)
            if S_text is None:
                S_text = np.zeros((sim_rows, sim_cols), dtype=float)
            elif S_text.shape != (sim_rows, sim_cols):
                S_safe = np.zeros((sim_rows, sim_cols), dtype=float)
                r = min(S_text.shape[0], sim_rows); c = min(S_text.shape[1], sim_cols)
                if r > 0 and c > 0:
                    S_safe[:r, :c] = S_text[:r, :c]
                S_text = S_safe
        
            PROFILE = "BALANCED"  # "CONTENT", "BALANCED", "TIMING_QC"
            if PROFILE == "CONTENT":      w_time, w_text, time_window_s = 0.4, 0.6, 90.0
            elif PROFILE == "BALANCED":   w_time, w_text, time_window_s = 0.6, 0.4, 45.0
            elif PROFILE == "TIMING_QC":  w_time, w_text, time_window_s = 0.85, 0.15, 25.0
            else:                         w_time, w_text, time_window_s = 0.6, 0.4, 45.0
        
            # WICHTIG: 3 Werte entgegennehmen!
            pairs, S, orphan_h = time_aware_alignment(
                _mdr, _ki,
                S_text=S_text,
                w_time=w_time,
                w_text=w_text,
                time_window_s=time_window_s
            )
        
            # --- 1: Zuordnungen prüfen: nutzt ein Hyp-Index (KI) mehrmals? ---  
            cnt = Counter([h for _, h, _ in pairs if h is not None])
            dup_h = {h for h, c in cnt.items() if c > 1}
            
            # --- 2: Pro KI-Segment nur das beste (höchste Score) behalten ---
            best_for_h = {}
            for r, h, s in pairs:
                if h is None:
                    continue
                if (h not in best_for_h) or (s > best_for_h[h][2]):
                    best_for_h[h] = (r, h, s)
            
            # --- 3: Alle anderen Duplikate auf „unmatched“ setzen ---
            new_pairs = []
            assigned_h = set()
            for r, h, s in pairs:
                if h is None:
                    new_pairs.append((r, None, 0.0))
                    continue
                if (h in best_for_h) and (best_for_h[h][0] == r) and (h not in assigned_h):
                    new_pairs.append((r, h, s))
                    assigned_h.add(h)
                else:
                    new_pairs.append((r, None, 0.0))
            
            pairs = new_pairs
            
            # (optional) kleine Statusmeldung
            if dup_h:
                msg.value = (msg.value + 
                    f"<div style='color:#8a6d3b'>Hinweis: {len(dup_h)} KI-Segmente waren mehrfach zugeordnet – "
                    f"nur jeweils der beste Treffer wurde behalten; übrige auf „kein Partner“ gesetzt.</div>")
        # -----------------------------------------------------------------------

        _progress_calc(0.25); _dbg_here("after_align", {"pairs": len(pairs)})

        # 4) Segmentmetriken (robust: greift sicher auf Texte/Token zu)
        rows = []
        for r_idx, h_idx, sim in pairs:
            r_tok = _safe_tok(ref_tok, r_idx)
            h_tok = _safe_tok(hyp_tok, h_idx) if h_idx is not None else []   # !!
            m = segment_metrics(r_tok, h_tok) if h_tok else {                 # !!
                "bleu_1":0,"bleu_2":0,"bleu_3":0,"bleu_4":0,
                "rouge1_precision":0,"rouge1_recall":0,"rouge1_f1":0,
                "rouge2_precision":0,"rouge2_recall":0,"rouge2_f1":0,
                "rougeL_precision":0,"rougeL_recall":0,"rougeL_f1":0,
                "ref_has_color":np.nan,"hyp_has_color":np.nan,
                "ref_has_motion":np.nan,"hyp_has_motion":np.nan,
                "len_ref":len(r_tok),"len_hyp":0,"syl_ref":sum(count_syllables_de_word(t) for t in r_tok),"syl_hyp":0
            }
        
            row = {
                "ref_idx": r_idx,
                "hyp_idx": (h_idx if h_idx is not None else np.nan),
                "alignment_cosine": (sim if h_idx is not None else 0.0),
                "ref_segment": _safe_txt(ref_segs, r_idx),
                "hyp_segment": (_safe_txt(hyp_segs, h_idx) if h_idx is not None else ""),
            }
            row.update(m)
            rows.append(row)
        df = pd.DataFrame(rows)
        _progress_calc(0.55); _dbg_here("after_segment_metrics", {"rows": len(df)})

        # ===== Timing-Metriken je gematchtem Paar =====
        tim_rows = []
        if _mdr is not None and _ki is not None and isinstance(pairs, list):
            for (r_idx, h_idx, _sim) in pairs:
                if h_idx is None:
                    tim_rows.append((np.nan, np.nan, np.nan, np.nan, 
                                     f"{_fmt_seconds(_mdr.iloc[r_idx]['start_sec'])}–{_fmt_seconds(_mdr.iloc[r_idx]['ende_sec'])}",
                                     ""))  # leere Hyp-Zeit
                    continue
                mr = _mdr.iloc[r_idx]; kh = _ki.iloc[int(h_idx)]
                overlap, d_start, d_end, d_mean = time_overlap_and_deltas(
                    mr.get("start_sec"), mr.get("ende_sec"),
                    kh.get("start_sec"), kh.get("ende_sec")
                )
                tim_rows.append((overlap, d_start, d_end, d_mean,
                                 f"{_fmt_seconds(mr.get('start_sec'))}–{_fmt_seconds(mr.get('ende_sec'))}",
                                 f"{_fmt_seconds(kh.get('start_sec'))}–{_fmt_seconds(kh.get('ende_sec'))}"))
            
            # In den Haupt-DataFrame 'df' integrieren (Reihenfolge entspricht 'pairs')
            df["time_overlap"] = [t[0] for t in tim_rows]
            df["Δstart_sec"]   = [t[1] for t in tim_rows]
            df["Δend_sec"]     = [t[2] for t in tim_rows]
            df["Δmean_sec"]    = [t[3] for t in tim_rows]
            df["zeit_ref"]     = [t[4] for t in tim_rows]
            df["zeit_hyp"]     = [t[5] for t in tim_rows]

            # Komfort-KPI: Timing Accuracy (0..1), fenstert bei 5s
            df["Timing Accuracy"] = np.clip(1.0 - (df["Δmean_sec"] / 5.0), 0.0, 1.0)
        else:
            # Falls kein mdr/ki vorhanden (sollte nicht passieren)
            for col in ["time_overlap","Δstart_sec","Δend_sec","Δmean_sec","zeit_ref","zeit_hyp","Timing Accuracy"]:
                df[col] = np.nan

        # 5) Optional: SBERT/BERTScore
        emb_ref = sentence_embeddings(df["ref_segment"].tolist()) if (_HAS_ST and not df.empty and __COMPUTE_SBERT__) else None
        emb_hyp = sentence_embeddings(df["hyp_segment"].tolist()) if (_HAS_ST and not df.empty and __COMPUTE_SBERT__) else None
        df["sbert_cosine"] = (emb_ref * emb_hyp).sum(axis=1) if (emb_ref is not None and emb_hyp is not None) else np.nan

        bs = bertscore_pairs(df["ref_segment"].tolist(), df["hyp_segment"].tolist()) if (_HAS_BERTSCORE and not df.empty and __COMPUTE_BERTSCORE__) else None
        df["bertscore_f1"] = bs if bs is not None else np.nan
        _progress_calc(0.75); _dbg_here("after_sbert_bertscore")

        # 6) RAGAS-Fallbacks + Tabelle
        ragas_seg = ragas_like_metrics(df["ref_segment"].tolist(), df["hyp_segment"].tolist(), df)
        _progress_calc(0.90); _dbg_here("after_ragas")

        # ----------------------------
        # Tabellenaufbereitung (Schritt 6a)
        # ----------------------------
        # Basis-Listen
        front = [c for c in ["ref_segment","hyp_segment"] if c in df.columns]
        pref  = [
            "alignment_cosine","sbert_cosine","bertscore_f1",
            "rougeL_f1","rouge2_f1","rouge1_f1",
            "bleu_4","bleu_3","bleu_2","bleu_1",
            "rougeL_precision","rougeL_recall","rouge2_precision","rouge2_recall","rouge1_precision","rouge1_recall",
            "len_ref","len_hyp","syl_ref","syl_hyp","ref_has_color","hyp_has_color","ref_has_motion","hyp_has_motion",
            "ref_idx","hyp_idx"
        ]

        # Zeitspalten aus den Original-Tabellen anhand der Indizes mappen
        if _mdr is not None and _ki is not None:
            def _safe_get(df_, i, col):
                try:
                    return df_.iloc[int(i)][col] if 0 <= int(i) < len(df_) else ""
                except Exception:
                    return ""
            df["start_ref"] = df["ref_idx"].map(lambda i: _safe_get(_mdr, i, "start"))
            df["ende_ref"]  = df["ref_idx"].map(lambda i: _safe_get(_mdr, i, "ende"))
            df["start_hyp"] = df["hyp_idx"].map(lambda i: _safe_get(_ki, int(i), "start") if pd.notna(i) else "")
            df["ende_hyp"]  = df["hyp_idx"].map(lambda i: _safe_get(_ki, int(i), "ende")  if pd.notna(i) else "")

        # Zeitspalten in die Front-Spalten aufnehmen (Zeit vor Text)
        time_cols = ["start_ref","ende_ref","start_hyp","ende_hyp"]
        front = [c for c in time_cols + ["ref_segment","hyp_segment"] if c in df.columns]

        # Timing-Metriken in die Anzeige übernehmen
        timing_cols = ["time_overlap","Δstart_sec","Δend_sec","Δmean_sec","Timing Accuracy"]

        # Spaltenauswahl (pref + timing)
        cols = front + [c for c in (pref + timing_cols) if c in df.columns]

        # Schöne Labels (Timing + Zeitfelder)
        rename_timing = {
            "time_overlap":    "Zeit-Overlap",
            "Δstart_sec":      "Δ-Start (s)",
            "Δend_sec":        "Δ-Ende (s)",
            "Δmean_sec":       "Δ-Mittel (s)",
            "Timing Accuracy": "Timing-Genauigkeit",
        }
        rename_times = {
            "start_ref": "Start Ref",
            "ende_ref":  "Ende Ref",
            "start_hyp": "Start Hyp",
            "ende_hyp":  "Ende Hyp",
        }

        table = df[cols].rename(columns={
            "ref_segment":"Referenzsegment (MDR)",
            "hyp_segment":"Hypothesensegment (KI)",
            "alignment_cosine":"Cosine (TF-IDF)",
            "sbert_cosine":"SBERT Cosine",
            "bertscore_f1":"BERTScore F1",
            "rouge1_precision":"ROUGE-1 P","rouge1_recall":"ROUGE-1 R","rouge1_f1":"ROUGE-1 F1",
            "rouge2_precision":"ROUGE-2 P","rouge2_recall":"ROUGE-2 R","rouge2_f1":"ROUGE-2 F1",
            "rougeL_precision":"ROUGE-L P","rougeL_recall":"ROUGE-L R","rougeL_f1":"ROUGE-L F1",
            "bleu_1":"BLEU-1","bleu_2":"BLEU-2","bleu_3":"BLEU-3","bleu_4":"BLEU-4",
            "len_ref":"Wörter Ref","len_hyp":"Wörter Hyp",
            "syl_ref":"Silben Ref","syl_hyp":"Silben Hyp",
            "ref_has_color":"Ref Farbdetail","hyp_has_color":"Hyp Farbdetail",
            "ref_has_motion":"Ref Bewegung","hyp_has_motion":"Hyp Bewegung",
            "ref_idx":"Ref-Index","hyp_idx":"Hyp-Index"
        } | rename_timing | rename_times)

        # Zusatzkennzahlen
        if "Wörter Ref" in table.columns and "Wörter Hyp" in table.columns:
            table["len_ratio"] = np.where(table["Wörter Ref"] > 0, table["Wörter Hyp"]/table["Wörter Ref"], np.nan)
        if "Silben Ref" in table.columns and "Silben Hyp" in table.columns:
            table["syl_ratio"] = np.where(table["Silben Ref"] > 0, table["Silben Hyp"]/table["Silben Ref"], np.nan)

        _progress_calc(0.90)

        # 7) UI
        try:
            rebuild_tabs()
        except Exception as e:
            _catch_and_show("rebuild_tabs", e)
            return
        __HAVE_DATA__ = True
        _dbg_here("after_rebuild_tabs")
        
        # --- Summary/Report separat & mit Stage-Fehlerbox ---
        try:
            with summary_out:
                summary_out.clear_output(wait=True)
                render_summary_and_report()
        except Exception as e:
            _catch_and_show("render_summary_and_report", e)
            return

        # --- Charts separat & mit Stage-Fehlerbox ---
        try:
            with charts_out:
                charts_out.clear_output(wait=True)
                draw_charts()
        except Exception as e:
            _catch_and_show("draw_charts", e)
            # Charts-Fehler sollen die Pipeline nicht komplett abbrechen:
            pass

        msg.value = "<div style='color:#2e7d32'>Berechnung abgeschlossen und Tabs aktualisiert.</div>"
        _progress_calc(1.0)

    except Exception as e:
        if "Stufe" not in msg.value:   # nichts überschreiben, wenn _catch_and_show schon sprach
            msg.value = f"<div style='color:#b00020'>Fehler in der Berechnung: {escape(str(e))}</div>"
        p_calc.bar_style = "danger"
    finally:
        __RUNNING__ = False
        _set_compute_enabled()   # aktiviert Button wieder, falls Daten da

# =======================
# ⚙️ Settings-Panel
# =======================
UI_TEXT_WIDTH = 250
UI_NUM_MIN    = 95
UI_NUM_MAX    = 115
UI_TIME_MIN   = 48
UI_TIME_MAX   = 56

def build_settings_panel():
    # Verfügbarkeiten
    lib_row = W.HBox([
        W.HTML(f"<b>spaCy:</b> {'✅' if _HAS_SPACY else '❌'}"),
        W.HTML(f"<b>SBERT:</b> {'✅' if _HAS_ST else '❌'}"),
        W.HTML(f"<b>BERTScore:</b> {'✅' if _HAS_BERTSCORE else '❌'}"),
        W.HTML(f"<b>RAGAS:</b> {'✅' if _ragas_available else '❌'}")
    ], layout=W.Layout(justify_content="space-between"))

    # Schalter
    chk_sbert     = W.Checkbox(value=bool(CONFIG.get("USE_SBERT", True) and _HAS_ST), description="SBERT berechnen", disabled=not _HAS_ST)
    chk_bertscore = W.Checkbox(value=bool(CONFIG.get("USE_BERTSCORE", True) and _HAS_BERTSCORE), description="BERTScore berechnen", disabled=not _HAS_BERTSCORE)
    chk_ragas     = W.Checkbox(value=bool(CONFIG.get("USE_RAGAS", True) and _ragas_available), description="RAGAS-Tab anzeigen", disabled=not _ragas_available)
    chk_spacy     = W.Checkbox(value=bool(CONFIG.get("USE_SPACY", True) and _HAS_SPACY), description="spaCy (Lemmata) nutzen", disabled=not _HAS_SPACY)

    # Ampel-Slider
    def _sl(label, key):
        y,g = THRESHOLDS[key]
        return (W.FloatSlider(value=y, min=0, max=1, step=0.01, description=f"{label} gelb"),
                W.FloatSlider(value=g, min=0, max=1, step=0.01, description=f"{label} grün"))

    s_tfidf_y, s_tfidf_g = _sl("TF-IDF", "Cosine (TF-IDF)")
    s_sbert_y, s_sbert_g = _sl("SBERT",  "SBERT Cosine")
    s_bert_y,  s_bert_g  = _sl("BERT F1","BERTScore F1")
    s_rL_y,    s_rL_g    = _sl("ROUGE-L","ROUGE-L F1")
    s_bleu_y,  s_bleu_g  = _sl("BLEU-4", "BLEU-4")

    # Ratio-Bänder & Darstellung
    rng_len = W.FloatRangeSlider(value=LEN_RATIO_BAND, min=0.5, max=1.5, step=0.01, description="len_ratio OK-Band")
    rng_syl = W.FloatRangeSlider(value=SYL_RATIO_BAND, min=0.5, max=1.5, step=0.01, description="syl_ratio OK-Band")
    s_text_w = W.IntSlider(value=UI_TEXT_WIDTH, min=220, max=700, step=10, description="Textbreite (px)")
    s_num_min = W.IntSlider(value=UI_NUM_MIN, min=60, max=150, step=5, description="Num min (px)")
    s_num_max = W.IntSlider(value=UI_NUM_MAX, min=70, max=180, step=5, description="Num max (px)")

    btn_apply = W.Button(description="Übernehmen & neu rendern", button_style="primary", icon="check")
    btn_export = W.Button(description="Auto-Report exportieren (Markdown)", icon="download")
    btn_export_txt = W.Button(description="Vergleich als .txt (TSV) exportieren", icon="download")
    toast = W.HTML("")

    sec_calc = W.VBox([lib_row, W.HBox([chk_sbert, chk_bertscore, chk_ragas, chk_spacy])])
    sec_thr  = W.VBox([s_tfidf_y, s_tfidf_g, s_sbert_y, s_sbert_g, s_bert_y, s_bert_g, s_rL_y, s_rL_g, s_bleu_y, s_bleu_g, W.HTML("<hr>"), rng_len, rng_syl])
    sec_disp = W.VBox([W.HBox([s_text_w, s_num_min, s_num_max])])
    sec_act  = W.VBox([W.HBox([btn_apply, btn_export, btn_export_txt]), toast])

    acc = W.Accordion(children=[sec_calc, sec_thr, sec_disp, sec_act])
    for i, title in enumerate(["Berechnung", "Ampel-Schwellen", "Darstellung", "Aktionen"]):
        acc.set_title(i, title)

    def _notify(msg, ok=True):
        color = "#d4edda" if ok else "#fdecea"
        border = "#28a745" if ok else "#c62828"
        t = datetime.now().strftime("%H:%M:%S")
        toast.value = f"<div style='margin-top:8px;padding:8px 10px;border:1px solid {border};border-radius:6px;background:{color}'>\
                        <b>{'✓' if ok else '!'}</b> {msg} <span style='opacity:.7'>(um {t})</span></div>"

    def on_apply(_):
        global LEN_RATIO_BAND, SYL_RATIO_BAND, UI_TEXT_WIDTH, UI_NUM_MIN, UI_NUM_MAX
        # Flags
        globals()["__COMPUTE_SBERT__"]     = bool(chk_sbert.value)
        globals()["__COMPUTE_BERTSCORE__"] = bool(chk_bertscore.value)
        globals()["__SHOW_RAGAS__"]        = bool(chk_ragas.value)
        CONFIG["USE_SPACY"]                = bool(chk_spacy.value)

        # Schwellen
        THRESHOLDS.update({
            "Cosine (TF-IDF)": (s_tfidf_y.value, s_tfidf_g.value),
            "SBERT Cosine":    (s_sbert_y.value, s_sbert_g.value),
            "BERTScore F1":    (s_bert_y.value,  s_bert_g.value),
            "ROUGE-L F1":      (s_rL_y.value,    s_rL_g.value),
            "BLEU-4":          (s_bleu_y.value,  s_bleu_g.value),
        })
        LEN_RATIO_BAND = tuple(rng_len.value)
        SYL_RATIO_BAND = tuple(rng_syl.value)
        UI_TEXT_WIDTH, UI_NUM_MIN, UI_NUM_MAX = s_text_w.value, s_num_min.value, s_num_max.value

        try:
            rebuild_tabs()
            _notify("Einstellungen übernommen und Tabs neu gerendert.")
            _scroll_top()
        except Exception as e:
            _notify(f"Einstellungen übernommen, aber Rebuild-Hinweis: {e}", ok=False)

    def on_export(_):
        try:
            ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            path = f"/mnt/data/ad_report_{ts}.md"
            with open(path, "w", encoding="utf-8") as f:
                f.write(report)
            _notify(f"Report gespeichert: <a href='sandbox:{path}' target='_blank'>{os.path.basename(path)}</a>")
            _scroll_top()
        except Exception as e:
            _notify(f"Export fehlgeschlagen: {e}", ok=False)

    def on_export_txt(_):
        try:
            import pandas as pd
            if not (globals().get("__HAVE_DATA__") and isinstance(globals().get("table"), pd.DataFrame)):
                raise RuntimeError("Keine Ergebnisse – bitte zuerst Berechnen.")
    
            base = globals()["table"].copy()
            rag = globals().get("ragas_seg", None)
            if isinstance(rag, pd.DataFrame) and len(rag) == len(base):
                base = pd.concat([base, rag], axis=1)
    
            # RAGAS Anzeige-Header zurück auf Basisnamen
            inv = {v: k for k, v in globals().get("RAGAS_DISPLAY_MAP", {}).items()}
            if inv: base = base.rename(columns=inv)
    
            export_cols_pref = [
                "Start Ref","Ende Ref","Referenzsegment (MDR)",
                "Start Hyp","Ende Hyp","Hypothesensegment (KI)",
                "Cosine (TF-IDF)","SBERT Cosine","BERTScore F1","ROUGE-L F1","BLEU-4",
                "ROUGE-1 P","ROUGE-1 R","ROUGE-2 P","ROUGE-2 R","ROUGE-L P","ROUGE-L R",
                "Wörter Ref","Wörter Hyp","len_ratio","Silben Ref","Silben Hyp","syl_ratio",
                "Ref Farbdetail","Hyp Farbdetail","Ref Bewegung","Hyp Bewegung",
                "RAGAS: semantic_similarity","RAGAS: answer_similarity","RAGAS: faithfulness",
                "RAGAS: answer_relevancy","RAGAS: coverage","RAGAS: conciseness","RAGAS: fluency(FRE)"
            ]
            export_cols = [c for c in export_cols_pref if c in base.columns]
            export_df = base[export_cols].copy() if export_cols else base.copy()
    
            out_txt, out_tsv16 = _export_txt_safely(export_df)
            link = f"<a href='sandbox:{out_txt}' target='_blank'>{os.path.basename(out_txt)}</a>"
            link2 = f" · Excel: <a href='sandbox:{out_tsv16}' target='_blank'>{os.path.basename(out_tsv16)}</a>" if out_tsv16 else ""
            _notify(f"TXT gespeichert: {link}{link2}")
            _scroll_top()
        except Exception as e:
            _notify(f"TXT-Export fehlgeschlagen: {e}", ok=False)

    btn_apply.on_click(on_apply)
    btn_export.on_click(on_export)
    btn_export_txt.on_click(on_export_txt)
    return acc

settings_widget = build_settings_panel()

# ====================================
# 🎨 Tabs (farbig) & Tabellen-Styling
# ====================================
display(HTML("""
<style>
.widget-tab .p-TabBar-content { padding: 2px 2px 0 2px; }
.widget-tab .p-TabBar-tab {
  margin-right: 6px; border: 1px solid #d7dbe0; border-bottom: none;
  border-radius: 8px 8px 0 0; padding: 6px 12px; font-size: 12px;
  background: #f6f7f9; color: #222;
}
.widget-tab .p-TabBar-tab:hover { filter: brightness(0.98); }
.widget-tab .p-TabBar-tab.p-mod-current { font-weight: 600; box-shadow: 0 -1px 0 0 #999 inset; }
.widget-tab .p-TabBar-tab:nth-child(1) { background:#E3F2FD; }
.widget-tab .p-TabBar-tab:nth-child(2) { background:#E8F5E9; }
.widget-tab .p-TabBar-tab:nth-child(3) { background:#FFF3E0; }
.widget-tab .p-TabBar-tab:nth-child(4) { background:#E0F7FA; }
.widget-tab .p-TabBar-tab:nth-child(5) { background:#F3E5F5; }
.widget-tab .p-TabBar-tab:nth-child(6) { background:#ECEFF1; }
.widget-tab .p-TabBar-tab:nth-child(7) { background:#FFFDE7; }
.widget-tab .p-TabBar-tab:nth-child(8) { background:#F1F8E9; }
</style>
"""))

TIME_COLS = ["Start Ref","Ende Ref","Start Hyp","Ende Hyp"]
TEXT_COLS = ["Referenzsegment (MDR)", "Hypothesensegment (KI)"]

def _make_view(cols_right):
    """
    Baut die Spaltenreihenfolge: erst Zeitspalten (schmal), dann Textspalten,
    dann die gewünschten KPI-Spalten (falls vorhanden). Duplikate werden
    entfernt, Reihenfolge bleibt stabil.
    """
    # gewünschte Grundreihenfolge
    cols = TIME_COLS + TEXT_COLS + [c for c in cols_right if c in table.columns]

    # Duplikate entfernen (Order-preserving)
    seen = set()
    cols = [c for c in cols if (c in table.columns) and (not (c in seen or seen.add(c)))]

    return table[cols].copy()

def _style_table(
    df_view: pd.DataFrame,
    text_width_px: int = UI_TEXT_WIDTH,
    num_min_px: int = UI_NUM_MIN,
    num_max_px: int = UI_NUM_MAX,
    font_px: int = 11,
    cell_pad: str = "6px 8px",
    wrap_header_prefixes: tuple[str, ...] = ("ROUGE-", "Silben", "Wörter", "RAGAS"),
    wrap_header_width_px: int = 78,
    col_widths: dict[str, int] | None = None,
    table_layout: str = "fixed",
    time_min_px: int = UI_TIME_MIN,
    time_max_px: int = UI_TIME_MAX,
):
    text_cols_local = [c for c in TEXT_COLS if c in df_view.columns]
    time_cols_local = [c for c in TIME_COLS if c in df_view.columns]
    num_cols_local = [c for c in df_view.columns if c not in text_cols_local + time_cols_local]

    for c in num_cols_local:
        df_view[c] = pd.to_numeric(df_view[c], errors="coerce")
        
    # Timing-Spalten explizit konvertieren (falls als String reinkommen)
    for col in ["Δ-Mittel (s)", "Δ-Start (s)", "Δ-Ende (s)", "Zeit-Overlap", "Timing-Genauigkeit"]:
        if col in df_view.columns:
            df_view[col] = pd.to_numeric(df_view[col], errors="coerce")

    float_cols = [c for c in num_cols_local if pd.api.types.is_float_dtype(df_view[c])]
    int_cols = [c for c in num_cols_local if pd.api.types.is_integer_dtype(df_view[c])]

    fmt_map = {c: "{:.2f}" for c in float_cols}
    for c in int_cols:
        fmt_map[c] = "{:.0f}"
    for c in ("Wörter Ref", "Wörter Hyp", "Silben Ref", "Silben Hyp"):
        if c in df_view.columns:
            fmt_map[c] = "{:.0f}"
    
    base_styles = [
        {"selector": "thead th", "props": [
            ("background", "#f7f9fc"),
            ("border-bottom", "2px solid #e0e0e0"),
            ("box-shadow", "0 2px 2px rgba(0,0,0,0.05)"),
            ("text-align", "right"),
        ]},
        {"selector": "th", "props": [
            ("font-size", f"{font_px}px"),
            ("padding", cell_pad),
            ("white-space", "normal"),
            ("vertical-align", "bottom"),
            ("vertical-align", "middle"),
        ]},
        {"selector": "td", "props": [
            ("font-size", f"{font_px}px"),
            ("padding", cell_pad),
            ("vertical-align", "top"),
        ]},
        {"selector": "tbody tr:nth-child(odd)",  "props": [("background", "#f7f9fc")]},
        {"selector": "tbody tr:nth-child(even)", "props": [("background", "#ffffff")]},
    ]

    for col_name in text_cols_local:
        i = df_view.columns.get_loc(col_name)
        base_styles.append({
            "selector": f"th.col_heading.level0.col{i}",
            "props": [("text-align", "left")],
        })

    for i, name in enumerate(df_view.columns):
        norm = _norm_col_key(name)
        if any(norm.startswith(p) for p in wrap_header_prefixes):
            base_styles.append({
                "selector": f"th.col_heading.level0.col{i}",
                "props": [
                    ("max-width", f"{wrap_header_width_px}px"),
                    ("white-space", "pre-line"),
                    ("word-break", "break-word"),
                    ("overflow-wrap", "anywhere"),
                    ("line-height", "1.15"),
                ],
            })
    _len_cols = {
        "Wörter Ref","Wörter Hyp","len_ratio",
        "Silben Ref","Silben Hyp","syl_ratio",
        "Ref Farbdetail","Hyp Farbdetail","Ref Bewegung","Hyp Bewegung",
    }
    for i, name in enumerate(df_view.columns):
        if name in _len_cols:
            base_styles.append({
                "selector": f"th.col_heading.level0.col{i}",
                "props": [
                    ("white-space", "nowrap"),
                    ("vertical-align", "middle"),
                ],
            })

    s = df_view.style
    s = s.format(fmt_map, na_rep="")
    s = s.set_table_styles(base_styles)
   
    cls = "ad-table ad-layout-fixed" if table_layout == "fixed" else "ad-table ad-layout-auto"
    s = s.set_table_attributes(f'class="{cls}"')

    # --- Header-Breiten an TD-Breiten koppeln ---
    # Zeitspalten (Monospace)
    for i, name in enumerate(df_view.columns):
        if name in time_cols_local:
            base_styles.append({
                "selector": f"th.col_heading.level0.col{i}",
                "props": [
                    ("min-width", f"{time_min_px}px"),
                    ("max-width", f"{time_max_px}px"),
                    ("white-space", "nowrap"),
                    ("text-align", "center"),
                    ("font-variant-numeric", "tabular-nums"),
                    ("font-family", "ui-monospace, SFMono-Regular, Menlo, Consolas, monospace"),
                ],
            })
    
    # Textspalten (links)
    for i, name in enumerate(df_view.columns):
        if name in text_cols_local:
            base_styles.append({
                "selector": f"th.col_heading.level0.col{i}",
                "props": [
                    ("min-width", f"{text_width_px}px"),
                    ("max-width", f"{text_width_px}px"),
                    ("white-space", "pre-line"),
                    ("word-break", "break-word"),
                    ("overflow-wrap", "anywhere"),
                    ("text-align", "left"),
                ],
            })
    
    # Numerische Spalten (Standard-Min/Max)
    for i, name in enumerate(df_view.columns):
        if name in num_cols_local:
            base_styles.append({
                "selector": f"th.col_heading.level0.col{i}",
                "props": [
                    ("min-width", f"{num_min_px}px"),
                    ("max-width", f"{num_max_px}px"),
                    ("white-space", "nowrap"),
                    ("text-align", "right"),
                ],
            })

    s = s.set_properties(
        subset=text_cols_local,
        **{
            "min-width": f"{text_width_px}px",
            "max-width": f"{text_width_px + 120}px",
            "white-space": "normal",
            "word-break": "break-word",
            "overflow-wrap": "anywhere",
            "text-align": "left",
        },
    )

    s = s.set_properties(
        subset=time_cols_local,
        **{
            "min-width": f"{time_min_px}px",
            "max-width": f"{time_max_px}px",
            "white-space": "nowrap",
            "text-align": "center",
            "font-variant-numeric": "tabular-nums",
            "font-family": "ui-monospace, SFMono-Regular, Menlo, Consolas, monospace",
        },
    )

    for i, name in enumerate(df_view.columns):
        if name in TEXT_COLS:
            base_styles.append({
                "selector": f"th.col_heading.level0.col{i}",
                "props": [
                    ("max-width", f"{text_width_px}px"),
                    ("white-space", "pre-line"),
                    ("word-break", "break-word"),
                    ("overflow-wrap", "anywhere"),
                    ("line-height", "1.15")
                ]
            })

    s = s.set_properties(
        subset=num_cols_local,
        **{
            "min-width": f"{num_min_px}px",
            "max-width": f"{num_max_px}px",
            "white-space": "nowrap",
            "text-align": "right",
        },
    )

    if col_widths:
        for col, w in col_widths.items():
            if col in df_view.columns:
                s = s.set_properties(subset=[col], **{"min-width": f"{w}px", "max-width": f"{w}px"})

    def _ampel_series(series: pd.Series):
        col = series.name
        return [f"background-color: {_color_for_kpi(col, v)}" for v in series]

    for c in list(num_cols_local) + [x for x in ("len_ratio", "syl_ratio") if x in df_view.columns]:
        s = s.apply(_ampel_series, axis=0, subset=[c])

    for c in [x for x in ("Hyp Farbdetail", "Hyp Bewegung") if x in df_view.columns]:
        s = s.apply(
            lambda ser: [
                (f"background-color: {COLOR_OK}" if bool(v) else f"background-color: {COLOR_BAD}") if pd.notna(v) else ""
                for v in ser
            ],
            axis=0,
            subset=[c],
        )

    try:
        s = s.hide(axis="index")
    except Exception:
        try:
            s = s.hide_index()
        except Exception:
            pass

    return s

# KPI-Gruppen
group_quality = ["Cosine (TF-IDF)", "SBERT Cosine", "BERTScore F1","ROUGE-L F1","BLEU-4"]
group_rouge   = ["ROUGE-L P","ROUGE-L R","ROUGE-2 P","ROUGE-2 R","ROUGE-1 P","ROUGE-1 R"]
group_length  = ["Wörter Ref","Wörter Hyp","len_ratio","Silben Ref","Silben Hyp","syl_ratio",
                 "Ref Farbdetail","Hyp Farbdetail","Ref Bewegung","Hyp Bewegung"]
group_align   = ["Ref-Index","Hyp-Index"]
group_timing = ["Start Ref","Ende Ref","Start Hyp","Ende Hyp", "Δ-Mittel (s)","Δ-Start (s)","Δ-Ende (s)","Zeit-Overlap","Timing-Genauigkeit"]

# Breiten-Mapping (gezielt)
rouge_widths = {"ROUGE-L P":70,"ROUGE-L R":70,"ROUGE-2 P":70,"ROUGE-2 R":70,"ROUGE-1 P":70,"ROUGE-1 R":70}
length_widths = {"Wörter Ref":70,"Wörter Hyp":70,"len_ratio":70,"Silben Ref":80,"Silben Hyp":80,"syl_ratio":70,
                 "Ref Farbdetail":80,"Hyp Farbdetail":80,"Ref Bewegung":80,"Hyp Bewegung":80}

def _text_grid_html(pairs, ref_segs, hyp_segs) -> str:
    if not isinstance(pairs, (list, tuple)) or len(pairs) == 0:
        return """
        <div class="adgrid2">
          <p><em>Noch keine Ausrichtung vorhanden. Bitte oben <b>Berechnen</b> ausführen.</em></p>
        </div>
        """

    rows = []
    for r_idx, h_idx, _ in pairs:
        # Referenz immer prüfen
        if not (isinstance(r_idx, int) and 0 <= r_idx < len(ref_segs)):
            continue
        rs = escape(ref_segs[r_idx])

        # Hyp kann None oder out-of-range sein → dann leer anzeigen
        if isinstance(h_idx, int) and 0 <= h_idx < len(hyp_segs):
            hs = escape(hyp_segs[h_idx])
        else:
            hs = ""  # kein Partner

        rows.append(f"""
          <div class="adgrid2-row">
            <div class="adgrid2-cell adgrid2-ref">{rs}</div>
            <div class="adgrid2-cell adgrid2-hyp">{hs}</div>
          </div>
        """)

    return """
    <style>
      .adgrid2 { width: 100%; }
      .adgrid2-header {
        position: sticky; top: var(--ad-top);
        z-index: 50; display: grid; grid-template-columns: 1fr 1fr; gap: 6px;
        background: #f7f9fc; border-bottom: 2px solid #e0e0e0; box-shadow: 0 2px 2px rgba(0,0,0,.05);
        padding: 6px 8px; font-weight: 600;
      }
      .adgrid2-row { display: grid; grid-template-columns: 1fr 1fr; gap: 6px; padding: 6px 8px; }
      .adgrid2-row:nth-child(odd)  { background: #f0f7ff; }
      .adgrid2-row:nth-child(even) { background: #ffffff; }
      .adgrid2-cell { white-space: normal; line-height: 1.35; word-break: break-word; overflow-wrap: anywhere; }
    </style>
    <div class="adgrid2" role="table" aria-label="Segmentierter Textvergleich">
      <div class="adgrid2-header" role="row">
        <div role="columnheader">Referenzsegment (MDR)</div>
        <div role="columnheader">Hypothesensegment (KI)</div>
      </div>
      <div class="adgrid2-body" role="rowgroup">
        """ + "".join(rows) + """
      </div>
    </div>
    """

def _panel(html: str) -> widgets.HTML:
    # X-Scroll-Hülle (nur horizontal), keine eigene Y-Scroll
    wrapped = f'<div class="ad-xscroll"><div class="ad-xinner">{html}</div></div>'
    return widgets.HTML(wrapped)

def rebuild_tabs():
    host_cls = f"ad-tabs-{int(time.time()*1000)}"
    text_html = _text_grid_html(pairs, ref_segs, hyp_segs)

    # ---------- 1) "RAGAS"- und "Alle"-Grundtabellen vorbereiten ----------
    _base = table[TIME_COLS + TEXT_COLS].copy()
    _rag  = ragas_seg if isinstance(ragas_seg, pd.DataFrame) else pd.DataFrame(index=_base.index)
    ragas_view = pd.concat([_base, _rag], axis=1).rename(columns=RAGAS_DISPLAY_MAP)

    table_all = (
        pd.concat([table, ragas_seg], axis=1)
        if isinstance(ragas_seg, pd.DataFrame) and len(ragas_seg) == len(table)
        else table.copy()
    )
    table_all_disp = table_all.rename(columns=RAGAS_DISPLAY_MAP)

    all_order = (
        TIME_COLS + TEXT_COLS
        + ["Cosine (TF-IDF)","SBERT Cosine","BERTScore F1","ROUGE-L F1","BLEU-4",
           "ROUGE-L P","ROUGE-L R","ROUGE-2 P","ROUGE-2 R","ROUGE-1 P","ROUGE-1 R"]
        + ["Wörter Ref","Wörter Hyp","Silben Ref","Silben Hyp","len_ratio","syl_ratio",
           "Ref Farbdetail","Hyp Farbdetail","Ref Bewegung","Hyp Bewegung"]
        + ["Ref-Index","Hyp-Index"]
        + ["RAGAS\nsemantic\nsimilarity","RAGAS\nanswer\nsimilarity","RAGAS\nfaithfulness",
           "RAGAS\nanswer\nrelevancy","RAGAS\ncoverage","RAGAS\nconciseness","RAGAS\nfluency\n(FRE)"]
    )
    sel = [c for c in all_order if c in table_all_disp.columns] or list(table_all_disp.columns)
    table_all_disp = table_all_disp[sel]

    # Breiten-Maps
    ragas_widths = {
        "RAGAS\nsemantic\nsimilarity": 92,
        "RAGAS\nanswer\nsimilarity":   150,
        "RAGAS\nfaithfulness":         92,
        "RAGAS\nanswer\nrelevancy":    92,
        "RAGAS\ncoverage":             92,
        "RAGAS\nconciseness":          92,
        "RAGAS\nfluency\n(FRE)":       96,
    }
    rouge_widths = {"ROUGE-L P":70,"ROUGE-L R":70,"ROUGE-2 P":70,"ROUGE-2 R":70,"ROUGE-1 P":70,"ROUGE-1 R":70}
    length_widths = {"Wörter Ref":70,"Wörter Hyp":70,"len_ratio":70,"Silben Ref":80,"Silben Hyp":80,"syl_ratio":70,
                     "Ref Farbdetail":80,"Hyp Farbdetail":80,"Ref Bewegung":80,"Hyp Bewegung":80}

    col_widths_all = {}
    col_widths_all.update(length_widths)
    col_widths_all.update({
        "RAGAS\nsemantic\nsimilarity": 110,
        "RAGAS\nanswer\nsimilarity":   120,
        "RAGAS\nfaithfulness":         110,
        "RAGAS\nanswer\nrelevancy":    110,
        "RAGAS\ncoverage":             110,
        "RAGAS\nconciseness":          110,
        "RAGAS\nfluency\n(FRE)":       120,
    })

    # ---------- 2) Gestylte Views ----------
    view_quality = _style_table(
        _make_view(["Cosine (TF-IDF)", "SBERT Cosine", "BERTScore F1", "ROUGE-L F1", "BLEU-4"]),
        text_width_px=UI_TEXT_WIDTH,
        num_min_px=UI_NUM_MIN,
        num_max_px=UI_NUM_MAX,
        time_min_px=UI_TIME_MIN,
        time_max_px=UI_TIME_MAX,
        wrap_header_prefixes=(),                # ← WICHTIG: nicht ("ROUGE-", …)
        col_widths={"ROUGE-L F1": 92},          # etwas breiter, damit sicher einzeilig
        table_layout="fixed"
    )
    view_rouge = _style_table(
        _make_view(["ROUGE-L P","ROUGE-L R","ROUGE-2 P","ROUGE-2 R","ROUGE-1 P","ROUGE-1 R"]),
        text_width_px=UI_TEXT_WIDTH,
        num_min_px=UI_NUM_MIN, 
        num_max_px=UI_NUM_MAX,
        time_min_px=UI_TIME_MIN, 
        time_max_px=UI_TIME_MAX,
        # WICHTIG: kein Header-Wrap für ROUGE:
        wrap_header_prefixes=(),                 # ← vorher: ("ROUGE-",)
        # Etwas breitere feste Spaltenbreiten:
        col_widths={"ROUGE-L P": 90, "ROUGE-L R": 90,
                    "ROUGE-2 P": 90, "ROUGE-2 R": 90,
                    "ROUGE-1 P": 90, "ROUGE-1 R": 90},
        table_layout="fixed"
    )
    view_length = _style_table(
        _make_view(["Wörter Ref","Wörter Hyp","len_ratio","Silben Ref","Silben Hyp","syl_ratio",
                    "Ref Farbdetail","Hyp Farbdetail","Ref Bewegung","Hyp Bewegung"]),
        text_width_px=UI_TEXT_WIDTH, num_min_px=UI_NUM_MIN, num_max_px=UI_NUM_MAX,
        time_min_px=UI_TIME_MIN, time_max_px=UI_TIME_MAX,
        col_widths=length_widths, table_layout="fixed"
    )
    view_align = _style_table(
        _make_view(["Ref-Index","Hyp-Index"]),
        text_width_px=UI_TEXT_WIDTH, num_min_px=UI_NUM_MIN, num_max_px=UI_NUM_MAX,
        time_min_px=UI_TIME_MIN, time_max_px=UI_TIME_MAX, table_layout="fixed"
    )
    view_timing = _style_table(
        _make_view(["Start Ref","Ende Ref","Start Hyp","Ende Hyp",
                    "Δ-Mittel (s)","Δ-Start (s)","Δ-Ende (s)","Zeit-Overlap","Timing-Genauigkeit"]),
        text_width_px=UI_TEXT_WIDTH, num_min_px=UI_NUM_MIN, num_max_px=UI_NUM_MAX,
        time_min_px=UI_TIME_MIN, time_max_px=UI_TIME_MAX, table_layout="fixed"
    )
    view_ragas = _style_table(
        ragas_view,
        text_width_px=UI_TEXT_WIDTH, num_min_px=UI_NUM_MIN, num_max_px=UI_NUM_MAX,
        time_min_px=UI_TIME_MIN, time_max_px=UI_TIME_MAX,
        wrap_header_prefixes=("RAGAS",), wrap_header_width_px=90,
        col_widths=ragas_widths, table_layout="fixed"
    )
    view_all = _style_table(
        table_all_disp,
        text_width_px=UI_TEXT_WIDTH,
        num_min_px=UI_NUM_MIN, num_max_px=UI_NUM_MAX,
        time_min_px=UI_TIME_MIN, time_max_px=UI_TIME_MAX,
        wrap_header_prefixes=("RAGAS",),   # ROUGE NICHT mehr umbrechen
        wrap_header_width_px=110,
        col_widths=col_widths_all,
        table_layout="fixed",
    )
    
    # ---------- 3) Tabs rendern ----------
    with tabs_out:
        tabs_out.clear_output(wait=True)

        titles = ["Text","Qualität","ROUGE","Länge & Abdeckung","Alignment","Timing"]
        text_tab = widgets.VBox(
            [_panel(text_html), charts_out, summary_out],
            layout=widgets.Layout(overflow_y='visible')
        )
        children = [
            text_tab,
            _panel(sticky_styler(view_quality)),
            _panel(sticky_styler(view_rouge)),
            _panel(sticky_styler(view_length)),
            _panel(sticky_styler(view_align)),
            _panel(sticky_styler(view_timing)),
        ]
        if globals().get("__SHOW_RAGAS__", True):
            children.append(_panel(sticky_styler(view_ragas))); titles.append("RAGAS")
    
        children.append(_panel(sticky_styler(view_all))); titles.append("Alle")
        children.append(settings_widget); titles.append("⚙️ Einstellungen")

        tabs = widgets.Tab(children=children)
        for i, t in enumerate(titles):
            tabs.set_title(i, t)

        # Host-Box erzeugen
        host_box = widgets.Box([tabs])
        host_box.layout.width = '100%'
        
        # Klassen setzen
        host_box.add_class('ad-tabs'); host_box.add_class(host_cls)
        
        # WICHTIG: Host ist der einzige Y-Scroller; X ist verborgen
        host_box.layout = widgets.Layout(
            max_height='72vh',
            overflow_y='auto',     # vertikales Scrollen auf dem Host
            overflow_x='hidden'    # keine eigene X-Scrollbar am Host
        )
        
        # Abstand zwischen tab Menu und Fortschrittsanzeige
        host_box.layout.margin = '20px 0 0 0'
        display(host_box)
                
        # ===== CSS =====
        display(HTML(f"""
            <style>
            /* ============================================
               1. Host: einziger vertikaler Scroll-Container
               ============================================ */
            .{host_cls},
            .{host_cls} > .widget-tab,
            .{host_cls} :is(.p-StackedPanel, .lm-StackedPanel) {{
              width: 100%;
              max-width: 100%;
              box-sizing: border-box;
            }}
            .{host_cls} {{
              --ad-top: 36px;                 /* Default-Offset für Sticky-Header */
              position: relative;
              max-height: 72vh;
              overflow-y: auto !important;    /* nur vertikales Scrollen im Host */
              overflow-x: hidden !important;  /* horizontal gesperrt */
              box-sizing: border-box;
            }}
            
            /* ==========================
               2. Tab-Leiste (sticky top)
               ========================== */
            .{host_cls} :is(.p-TabBar, .lm-TabBar) {{
              position: sticky;
              top: 0;
              z-index: 1000;
              background: #fff;
              border-bottom: 1px solid #e6e9ef;
              overflow-x: auto !important;
              overflow-y: hidden !important;
              white-space: nowrap;
              max-width: 100%;
              -webkit-overflow-scrolling: touch;
              scrollbar-gutter: stable both-edges;
            }}
            .{host_cls} :is(.p-TabBar-content, .lm-TabBar-content) {{
              display: inline-flex !important;
              flex-wrap: nowrap !important;
            }}
            .{host_cls} :is(.p-TabBar-tab, .lm-TabBar-tab) {{
              flex: 0 0 auto !important;
              max-width: max-content !important;
              padding: 6px 10px;
            }}
            
            /* ==========================
               3. Panels selbst: kein Scrollen
               ========================== */
            .{host_cls} :is(.p-StackedPanel, .lm-StackedPanel) {{
              overflow: visible !important;
            }}
            
            /* =====================================
               4. X-Scroll-Hülle für breite Tabellen
               ===================================== */
            .{host_cls} .ad-xscroll {{
              position: relative;
              width: 100%;
              max-width: 100%;
              overflow-x: auto;
              overflow-y: visible;
              padding-top: 2px;
              box-sizing: border-box;
            }}
            .{host_cls} .ad-xinner {{
              display: inline-block;
              width: auto;
              max-width: none;
              min-width: 100%;
              box-sizing: border-box;
            }}
            .{host_cls} .ad-xscroll table {{
              display: table;
              table-layout: auto !important;
              width: max-content;
              min-width: 100%;
              border-collapse: separate;
              border-spacing: 0;
            }}
            
            /* ===========================================
               5. Text-Grid (z. B. Referenz/Hypo): Header
               =========================================== */
            .{host_cls} .adgrid2-header {{
              position: sticky;
              top: var(--ad-top, 36px);
              z-index: 60;
              background: #f7f9fc;
              border-bottom: 2px solid #e0e0e0;
              box-shadow: 0 2px 2px rgba(0,0,0,.05);
            }}
            
            /* =================================
               6. Tabellen: Kopfzeile sticky
               ================================= */
            .{host_cls} table.dataframe {{
              border-collapse: separate;
              border-spacing: 0;
            }}
            .{host_cls} table.dataframe thead th {{
              position: sticky;
              top: var(--ad-top, 36px);
              z-index: 60;
              background: #f7f9fc;
              border-bottom: 2px solid #e0e0e0;
              box-shadow: 0 2px 2px rgba(0,0,0,.05);
            
              white-space: normal !important;
              word-break: normal !important;
              overflow-wrap: normal !important;
            
              line-height: 1.15;
              padding: 6px 8px;
              text-align: right;
            }}
            .{host_cls} table.dataframe thead tr {{
              position: sticky;
              top: var(--ad-top, 36px);
              z-index: 61; /* über th */
              background: #f7f9fc;
            }}
            
            /* =================================
               7. Tabellenkörper: freier Umbruch
               ================================= */
            .{host_cls} table.dataframe td {{
              overflow: visible !important;
              word-break: break-word !important;
              overflow-wrap: anywhere !important;
              white-space: pre-line !important;
              hyphens: auto;
              line-height: 1.35;
              vertical-align: top;
            }}
            </style>
            """))
        
        # ===== dynamischer Sticky-Offset =====
        display(Javascript(f"""
        (function(){{
          const host = document.querySelector('.{host_cls}');
          if (!host) return;
          const tabBar = () => host.querySelector('.p-TabBar, .lm-TabBar');
          function applyOffsets(){{
            const bar = tabBar();
            const h = Math.ceil(bar ? bar.getBoundingClientRect().height : 0);
            host.style.setProperty('--ad-top', h + 'px');
            host.querySelectorAll('.adgrid2-header, table.dataframe thead th')
                .forEach(th => th.style.top = h + 'px');
          }}
          requestAnimationFrame(applyOffsets);
          if (window.ResizeObserver) {{
            const ro = new ResizeObserver(applyOffsets);
            ro.observe(host);
            const bar = tabBar();
            if (bar) ro.observe(bar);
          }}
          window.addEventListener('resize', applyOffsets, {{ passive: true }});
        }})();
        """))
                
# ========================================================
# 📈 Visualisierungen – nur rendern, wenn Daten vorhanden
# ========================================================
ROW_FIGSIZE = (16, 5.8)
TITLE_SIZE  = 16
LABEL_SIZE  = 13
TICK_SIZE   = 12
GRID_ALPHA  = 0.25
DOT_SIZE    = 42
LINEWIDTH   = 1.6

plt.rcParams.update({
    "axes.grid": True,
    "grid.alpha": GRID_ALPHA,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlepad": 10.0,
    "axes.labelsize": LABEL_SIZE,
    "axes.titlesize": TITLE_SIZE,
    "xtick.labelsize": TICK_SIZE,
    "ytick.labelsize": TICK_SIZE,
})

def _nice_bins(vals):
    arr = np.asarray(vals, dtype=float).ravel()
    arr = arr[~np.isnan(arr)]
    if arr.size <= 1:
        return 5
    # >>> neu: nan-sichere Perzentile + Fallback
    try:
        q75, q25 = np.nanpercentile(arr, [75, 25])
    except Exception:
        return 5
    iqr = max(q75 - q25, 1e-9)
    binw = 2 * iqr / (arr.size ** (1/3))
    bins = int(np.ceil((arr.max() - arr.min()) / max(binw, 1e-12)))
    return int(np.clip(bins, 5, 20))

def _fix_01_axis(ax, vals):
    vmin = np.nanmin(vals); vmax = np.nanmax(vals)
    if vmin >= -0.05 and vmax <= 1.05: ax.set_xlim(0, 1)

def _annotate_mean(ax, mean_val):
    ymax = ax.get_ylim()[1]
    ax.axvline(mean_val, linestyle="--", linewidth=LINEWIDTH)
    ax.text(mean_val, ymax*0.95, f"μ = {mean_val:.2f}",
            ha="right", va="top",
            bbox=dict(facecolor="white", alpha=0.85, boxstyle="round,pad=0.2"))

def _hist(ax, series, title, xlabel):
    # >>> neu: alles kompromisslos auf 1D-Float bringen
    vals = pd.to_numeric(pd.Series(series).astype(object), errors="coerce").to_numpy(dtype=float).ravel()
    vals = vals[~np.isnan(vals)]
    if vals.size == 0:
        ax.set_axis_off()
        ax.text(0.5, 0.5, "Keine Daten", ha="center", va="center", fontsize=LABEL_SIZE)
        return
    ax.hist(vals, bins=_nice_bins(vals), edgecolor="white")
    _fix_01_axis(ax, vals)
    _annotate_mean(ax, float(np.mean(vals)))
    _plain_axes(ax)
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel("Häufigkeit")

def _pair_rows(panels):
    for i in range(0, len(panels), 2):
        fig, axes = plt.subplots(1, 2, figsize=ROW_FIGSIZE, constrained_layout=True)
        panels[i](axes[0])
        if i+1 < len(panels): panels[i+1](axes[1])
        else: axes[1].set_visible(False)
        plt.show()
        display(HTML("<div style='height:16px'></div>"))

def draw_charts():
    """Charts nur zeichnen, wenn bereits berechnet wurde."""
    plt.close('all')
    if not (globals().get("__HAVE_DATA__") and "df" in globals() and isinstance(df, pd.DataFrame) and not df.empty):
        display(HTML("<em>Noch keine Charts – bitte zuerst <b>Berechnen</b>.</em>"))
        return
    try:
        panels = []
        panels.append(lambda ax: _hist(ax, df.get("alignment_cosine", []), "TF-IDF Cosine – Verteilung", "Cosine"))
        panels.append(lambda ax: _hist(ax, df.get("rougeL_f1", []),        "ROUGE-L F1 – Verteilung",    "ROUGE-L F1"))
        if "bertscore_f1" in df.columns:  panels.append(lambda ax: _hist(ax, df["bertscore_f1"], "BERTScore F1 – Verteilung", "BERTScore F1"))
        if "sbert_cosine" in df.columns:  panels.append(lambda ax: _hist(ax, df["sbert_cosine"], "SBERT Cosine – Verteilung", "Cosine"))
        panels.append(lambda ax: _hist(ax, df.get("bleu_4", []),           "BLEU-4 – Verteilung",        "BLEU-4"))
        if "len_ratio" in table.columns:   panels.append(lambda ax: _hist(ax, table["len_ratio"], "Längenverhältnis (Wörter) – Verteilung", "len_hyp / len_ref"))
        if "syl_ratio" in table.columns:   panels.append(lambda ax: _hist(ax, table["syl_ratio"], "Längenverhältnis (Silben) – Verteilung", "syl_hyp / syl_ref"))

        if globals().get("__SHOW_RAGAS__") and "ragas_seg" in globals() and isinstance(ragas_seg, pd.DataFrame):
            for col, title in [
                ("RAGAS: semantic_similarity", "RAGAS: Semantic Similarity"),
                ("RAGAS: faithfulness",        "RAGAS: Faithfulness"),
                ("RAGAS: answer_relevancy",    "RAGAS: Answer Relevancy"),
            ]:
                if col in ragas_seg.columns:
                    panels.append(lambda ax, c=col, t=title: _hist(ax, ragas_seg[c], t, "Score"))

        # ==== Timing-Plots ====
        def _panel_timing_hist(ax):
            vals = pd.to_numeric(pd.Series(df.get("Δmean_sec", [])).astype(object), errors="coerce").to_numpy(dtype=float).ravel()
            vals = vals[~np.isnan(vals)]
            if vals.size == 0:
                ax.set_axis_off(); ax.text(0.5,0.5,"Keine Timing-Daten",ha="center",va="center"); return
            ax.hist(vals, bins=_nice_bins(vals), edgecolor="white")
            ax.set_title("Verteilung Δ-Mittel (s)")
            ax.set_xlabel("Sekunden"); ax.set_ylabel("Häufigkeit")

        def _panel_timing_scatter(ax):
            x = pd.to_numeric(pd.Series(df.get("Δmean_sec", [])).astype(object), errors="coerce").to_numpy(dtype=float).ravel()
            y = pd.to_numeric(pd.Series(df.get("alignment_cosine", [])).astype(object), errors="coerce").to_numpy(dtype=float).ravel()
            m = (~np.isnan(x)) & (~np.isnan(y))
            if m.sum() == 0:
                ax.set_axis_off(); ax.text(0.5,0.5,"Keine Daten",ha="center",va="center"); return
            ax.scatter(x[m], y[m], alpha=0.6)
            ax.set_title("Δ-Mittel (s) vs. Textähnlichkeit")
            ax.set_xlabel("Δ-Mittel (s)"); ax.set_ylabel("Cosine (TF-IDF)")
            ax.axvline(2, linestyle="--", linewidth=1)
            ax.axvline(5, linestyle="--", linewidth=1)
            _plain_axes(ax)
            ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
            ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

        _pair_rows([_panel_timing_hist, _panel_timing_scatter])
        _pair_rows(panels)
    except Exception as e:
        # Fehler nur im Chart-Bereich anzeigen – die Pipeline soll NICHT fehlschlagen
        display(HTML(f"<div style='color:#b00020'>Chart-Rendering übersprungen: {escape(str(e))}</div>"))

# =======================
# Summary & Report
# =======================
def render_summary_and_report():
    """Aggregationen, Rubrics & Auto-Report nur mit Daten rendern (robust)."""
    import numpy as _np
    import pandas as _pd

    # — 0) Vorbedingungen -------------------------------------------------------
    if not (
        globals().get("__HAVE_DATA__")
        and "df" in globals() and isinstance(df, _pd.DataFrame) and not df.empty
    ):
        return

    # — 1) Aggregation (Summary-Tabelle) ---------------------------------------
    metric_cols = df.select_dtypes(include=["number"]).columns.tolist()
    if not metric_cols:
        display(HTML("<em>Keine numerischen Metrikspalten gefunden.</em>"))
        return

    summary = pd.DataFrame({
        "n":          df[metric_cols].count(),
        "Mittelwert": df[metric_cols].mean(numeric_only=True),
        "StdAbw":     df[metric_cols].std(numeric_only=True, ddof=0),
        "Median":     df[metric_cols].median(numeric_only=True),
        "Min":        df[metric_cols].min(numeric_only=True),
        "Max":        df[metric_cols].max(numeric_only=True),
    })
    summary["Mittelwert"] = _pd.to_numeric(summary["Mittelwert"], errors="coerce")
    summary["StdAbw"]     = _pd.to_numeric(summary["StdAbw"], errors="coerce")
    summary["CV %"] = _np.where(summary["Mittelwert"] > 0,
                                100 * summary["StdAbw"] / summary["Mittelwert"], _np.nan)

    rename_map = {
        "alignment_cosine":"Cosine (TF-IDF)",
        "sbert_cosine":"SBERT Cosine",
        "bertscore_f1":"BERTScore F1",
        "rouge1_precision":"ROUGE-1 P","rouge1_recall":"ROUGE-1 R","rouge1_f1":"ROUGE-1 F1",
        "rouge2_precision":"ROUGE-2 P","rouge2_recall":"ROUGE-2 R","rouge2_f1":"ROUGE-2 F1",
        "rougeL_precision":"ROUGE-L P","rougeL_recall":"ROUGE-L R","rougeL_f1":"ROUGE-L F1",
        "bleu_1":"BLEU-1","bleu_2":"BLEU-2","bleu_3":"BLEU-3","bleu_4":"BLEU-4",
        "len_ref":"Wörter Ref","len_hyp":"Wörter Hyp",
        "syl_ref":"Silben Ref","syl_hyp":"Silben Hyp",
    }
    summary = summary.rename(index=rename_map)

    order = [
        "Cosine (TF-IDF)", "SBERT Cosine", "BERTScore F1", "ROUGE-L F1", "BLEU-4",
        "ROUGE-L P","ROUGE-L R","ROUGE-2 P","ROUGE-2 R","ROUGE-1 P","ROUGE-1 R",
        "Wörter Ref","Wörter Hyp","Silben Ref","Silben Hyp",
    ]
    order = [i for i in order if i in summary.index] + [i for i in summary.index if i not in order]
    summary = summary.loc[order]

    has_mean = "Mittelwert" in summary.columns and summary["Mittelwert"].notna().any()
    has_cv   = "CV %"       in summary.columns and summary["CV %"].notna().any()

    styler = (
        summary.style
        .format({
            "n": "{:.0f}", "Mittelwert": "{:.2f}", "StdAbw": "{:.2f}",
            "Median": "{:.2f}", "Min": "{:.2f}", "Max": "{:.2f}", "CV %": "{:.0f}"
        }, na_rep="")
        .set_table_styles([
            {"selector":"th","props":[("font-size","12px"),("text-align","left")]},
            {"selector":"td","props":[("font-size","12px")]}
        ])
    )
    if has_mean: styler = styler.background_gradient(subset=["Mittelwert"], cmap="Greens")
    if has_cv:   styler = styler.background_gradient(subset=["CV %"],       cmap="Reds")

    try:
        _left_html = styler.to_html()
    except Exception:
        _left_html = summary.to_html()

    # — 2) RAGAS-Summary (Fallback-ähnlich) ------------------------------------
    def _build_ragas_summary_df(ragas_seg_df):
        if not isinstance(ragas_seg_df, _pd.DataFrame) or ragas_seg_df.empty:
            return _pd.DataFrame(columns=["Mittelwert"])
        m = ragas_seg_df.mean(numeric_only=True).rename("Mittelwert").to_frame()
        try:    m = m.rename(index=RAGAS_DISPLAY_MAP)  # nur Anzeige
        except Exception: pass
        return m

    try:
        ragas_summary = _build_ragas_summary_df(globals().get("ragas_seg", None))
        _right_html = (
            ragas_summary.style.format("{:.2f}").to_html()
            if not ragas_summary.empty else "<em>Keine RAGAS-Ergebnisse vorhanden.</em>"
        )
    except Exception:
        _right_html = "<em>RAGAS-Zusammenfassung nicht verfügbar.</em>"

    display(HTML(
        f"<div style='display:grid;grid-template-columns:1fr 1fr;gap:16px;align-items:start'>"
        f"  <div>{_left_html}</div><div>{_right_html}</div>"
        f"</div>"
    ))

    # — 3) Mikro-Scores (über alle Tokens) ------------------------------------
    try:
        ref_all = [t for toks in ref_tok for t in toks]
        hyp_all = [t for toks in hyp_tok for t in toks]
    
        micro = {}
        micro["BLEU-4 (micro)"] = bleu_score(ref_all, hyp_all, max_n=4)
        micro["ROUGE-L F1 (micro)"] = rouge_l(ref_all, hyp_all)["f1"]
    
        # KORREKT: TF-IDF nur einmal bauen und Cosine zwischen den beiden Zeilen ziehen
        tfidf, _ = build_tfidf_vectors([ref_all, hyp_all])   # shape: (2, V)
        cos = float(cosine_similarity_matrix(tfidf[:1], tfidf[1:])[0, 0])
        micro["Cosine (TF-IDF, micro)"] = cos
    
        micro_row = _pd.DataFrame(micro, index=["Gesamt"])
        display(micro_row.style.format("{:.2f}"))
    except Exception as _e:
        # Zur Not still weiterlaufen, aber Fehler sichtbar machen
        display(HTML(f"<div style='color:#b00020'>Micro-Scores übersprungen: {escape(str(_e))}</div>"))

    # — 4) Rubrics + Timing ----------------------------------------------------
    def z01(x): 
        try: return float(_np.clip(x, 0, 1))
        except Exception: return _np.nan

    # Kennzahlen aus Tabellen holen
    pct = lambda s: float(_np.nanmean(_pd.to_numeric(s, errors="coerce"))*100.0) if len(s) else _np.nan
    ref_color = pct(table["Ref Farbdetail"]) if "Ref Farbdetail" in table else _np.nan
    hyp_color = pct(table["Hyp Farbdetail"]) if "Hyp Farbdetail" in table else _np.nan
    ref_move  = pct(table["Ref Bewegung"])  if "Ref Bewegung"  in table else _np.nan
    hyp_move  = pct(table["Hyp Bewegung"])  if "Hyp Bewegung"  in table else _np.nan
    color_gap = max(0.0, (ref_color - hyp_color)/100.0) if (_np.isfinite(ref_color) and _np.isfinite(hyp_color)) else _np.nan
    move_gap  = max(0.0, (ref_move  - hyp_move )/100.0) if (_np.isfinite(ref_move)  and _np.isfinite(hyp_move )) else _np.nan

    len_ratio_mean = float(_np.nanmean(table["len_ratio"])) if "len_ratio" in table else _np.nan
    syl_ratio_mean = float(_np.nanmean(table["syl_ratio"])) if "syl_ratio" in table else _np.nan
    rougeL_mean    = float(_np.nanmean(df["rougeL_f1"]))    if "rougeL_f1" in df   else _np.nan
    sbert_mean     = float(_np.nanmean(df["sbert_cosine"])) if "sbert_cosine" in df else _np.nan

    FRE_hyp = flesch_amstad(" ".join(globals().get("hyp_segs", [])))
    FRE_score = z01(FRE_hyp/100.0)

    def score_conciseness(r):
        if _np.isnan(r): return _np.nan
        return float((1 - min(1.0, abs(r-1.0))) * 100)

    def score_clarity(gap):
        if _np.isnan(gap): return _np.nan
        return float((1 - _np.clip(gap, 0, 1)) * 100)

    def score_alignment(rougeL, sbert):
        base = _np.nanmean([z01(rougeL), z01(sbert)])
        return float((base if _np.isfinite(base) else _np.nan) * 100)

    def score_readability(fre_norm):
        return float((fre_norm if _np.isfinite(fre_norm) else _np.nan) * 100)

    rows_rub = [
        ["Stil/Konzision (Wörter)" , score_conciseness(len_ratio_mean), "Nähe der Länge (Wörter) zu 1.0"],
        ["Stil/Konzision (Silben)" , score_conciseness(syl_ratio_mean), "Nähe der Länge (Silben) zu 1.0"],
        ["Lesehärte (FRE, Hyp)"    , score_readability(FRE_score)     , "Flesch–Amstad; höher = leichter"],
        ["Visuelle Klarheit"       , score_clarity(color_gap)          , "Gap Farbdetails Hyp vs. Ref (klein ist gut)"],
        ["Handlungsführung"        , score_clarity(move_gap)           , "Gap Bewegungsverben Hyp vs. Ref (klein ist gut)"],
        ["Inhaltsdeckung/Kohärenz" , score_alignment(rougeL_mean, sbert_mean),
                                      "ROUGE-L F1 & SBERT Cosine (Mittel)"],
    ]

    # Timing-Metriken
    try:
        dmean = float(_np.nanmean(_pd.to_numeric(df.get("Δmean_sec", _np.nan), errors="coerce")))
    except Exception: dmean = _np.nan
    try:
        within5 = _pd.to_numeric(df.get("Δmean_sec", _np.nan), errors="coerce") <= 5.0
        share_ok_5s = float(_np.nanmean(within5.astype(float))) if hasattr(within5, "notna") and within5.notna().any() else _np.nan
    except Exception: share_ok_5s = _np.nan
    try:
        timing_acc_mean = float(_np.nanmean(_pd.to_numeric(df.get("Timing Accuracy", _np.nan), errors="coerce")))
    except Exception: timing_acc_mean = _np.nan

    rows_rub.append([
        "Timing-Genauigkeit",
        float((timing_acc_mean if _np.isfinite(timing_acc_mean) else _np.nan) * 100.0),
        "1 - Δ-Mittel/5s (geclippt)"
    ])

    RUBRICS = _pd.DataFrame(rows_rub, columns=["Rubrik","Score (0–100)","Begründung"])
    RUBRICS["Score (0–100)"] = _pd.to_numeric(RUBRICS["Score (0–100)"], errors="coerce").clip(0, 100)

    # hübsche Anzeige
    def _score_bar_html(v: float) -> str:
        if _np.isnan(v): return "—"
        p = int(_np.clip(round(float(v)), 0, 100))
        return (
            f"<div style='display:flex;align-items:center;gap:.5rem'>"
            f"  <div style='flex:1;background:#eef5ee;border-radius:10px;height:18px;position:relative;overflow:hidden'>"
            f"    <div style='width:{p}%;height:100%;background:linear-gradient(90deg,#C8E6C9,#81C784)'></div>"
            f"  </div>"
            f"  <div style='min-width:32px;text-align:right;font-variant-numeric:tabular-nums'>{p}</div>"
            f"</div>"
        )
    def _chip_html(v: float) -> str:
        if _np.isnan(v): return ""
        s = float(v)
        if s >= 85:  label,bg,bd,fg = "stark","#E8F5E9","#66BB6A","#2E7D32"
        elif s >= 70:label,bg,bd,fg = "ok",   "#FFF8E1","#FFB300","#8C6D1F"
        else:        label,bg,bd,fg = "prüfen","#FFEBEE","#E57373","#B71C1C"
        return f"<span style='padding:2px 8px;border:1px solid {bd};border-radius:999px;background:{bg};color:{fg};font-size:12px'>{label}</span>"

    def show_rubrics_pretty(rubrics_df: _pd.DataFrame, overall_score: float | int | None = None):
        vis = rubrics_df.copy()
        vis.insert(1, "Score", vis["Score (0–100)"].map(_score_bar_html))
        vis.insert(2, "Bewertung", vis["Score (0–100)"].map(_chip_html))
        vis = vis.drop(columns=["Score (0–100)"])
        base_styles = [
            {"selector":"th","props":[("font-size","13px"),("text-align","left"),
                                      ("padding","6px 10px"),("border-bottom","1px solid #ddd")]},
            {"selector":"td","props":[("font-size","13px"),("padding","6px 10px"),("vertical-align","middle")]},
            {"selector":"tbody tr:nth-child(odd)", "props":[("background","#f7f9fc")]},
            {"selector":"tbody tr:nth-child(even)","props":[("background","#ffffff")]},
        ]
        sty = (vis.style
               .set_table_styles(base_styles)
               .set_properties(subset=["Rubrik","Begründung"], **{"white-space":"normal"})
               .format(na_rep=""))
        try:    sty = sty.hide(axis="index")
        except: 
            try: sty = sty.hide_index()
            except: pass
        display(HTML("<h4 style='margin:6px 0 6px 0'>MDR-Rubrics (0–100, höher besser)</h4>"))
        try:    html = sty.to_html(escape=False)
        except TypeError: html = sty.to_html()
        display(HTML(html))
        if overall_score is not None and not _np.isnan(overall_score):
            overall_int = int(round(float(overall_score)))
            display(HTML(
                f"<div style='margin-top:.6rem'><strong>Gesamt (gewichtet):</strong> "
                f"<span style='display:inline-block;margin-left:.35rem;padding:.15rem .6rem;"
                f"border:1px solid #66BB6A;border-radius:999px;background:#E8F5E9;"
                f"color:#2E7D32;font-weight:600'>{overall_int} / 100</span></div>"
            ))

    weights = {
        "Stil/Konzision (Wörter)" : 1.0,
        "Stil/Konzision (Silben)" : 1.0,
        "Lesehärte (FRE, Hyp)"    : 1.0,
        "Visuelle Klarheit"       : 1.0,
        "Handlungsführung"        : 1.0,
        "Inhaltsdeckung/Kohärenz" : 1.0,
        "Timing-Genauigkeit"      : 1.0,
    }
    num = den = 0.0
    for rubrik, score, _ in rows_rub:
        if _np.isfinite(score):
            w = weights.get(rubrik, 1.0)
            num += score * w; den += w
    overall = (num / den) if den else _np.nan

    show_rubrics_pretty(RUBRICS, overall)

    # Timing-Block (kurze Textausgabe)
    timing_md = (
        f"<h4 style='margin:10px 0 6px 0'>Timing</h4>"
        f"<ul style='margin:0 0 0 1.2rem;padding:0'>"
        f"<li>Mittlere zeitliche Abweichung (Δ-Mittel): "
        f"<strong>{'—' if _np.isnan(dmean) else f'{dmean:.2f} s'}</strong></li>"
        f"<li>Segmente innerhalb ±5 s: "
        f"<strong>{'—' if _np.isnan(share_ok_5s) else f'{share_ok_5s*100.0:.1f}%'} </strong></li>"
        f"<li>Timing-Genauigkeit (1 − Δ/5s, geclippt): "
        f"<strong>{'—' if _np.isnan(timing_acc_mean) else f'{timing_acc_mean*100.0:.1f}%'} </strong></li>"
        f"</ul>"
    )
    display(HTML(timing_md))

    # — 5) Empfehlungen + Report (Markdown) -----------------------------------
    def _safe(summary_df, row, col="Mittelwert"):
        try:
            return summary_df.loc[row, col]
        except Exception:
            return _np.nan

    def _level(val, thr):
        if _np.isnan(val) or thr is None: return "—"
        y, g = thr
        return "hoch" if val >= g else ("mittel" if val >= y else "niedrig")

    def _stability(cv):
        if _np.isnan(cv): return "—"
        return "sehr stabil" if cv <= 20 else ("moderat" if cv <= 40 else "volatil")

    cos_mean = _safe(summary, "Cosine (TF-IDF)"); cos_cv = _safe(summary, "Cosine (TF-IDF)", "CV %")
    sbert_m  = _safe(summary, "SBERT Cosine");    sbert_cv = _safe(summary, "SBERT Cosine", "CV %")
    bert_m   = _safe(summary, "BERTScore F1");    bert_cv  = _safe(summary, "BERTScore F1", "CV %")
    rL_m     = _safe(summary, "ROUGE-L F1");      rL_cv    = _safe(summary, "ROUGE-L F1", "CV %")
    bleu4_m  = _safe(summary, "BLEU-4");          bleu4_cv = _safe(summary, "BLEU-4", "CV %")

    w_ref = _safe(summary, "Wörter Ref"); w_hyp = _safe(summary, "Wörter Hyp")
    s_ref = _safe(summary, "Silben Ref"); s_hyp = _safe(summary, "Silben Hyp")
    len_ratio = (w_hyp / w_ref) if (_np.isfinite(w_ref) and w_ref > 0) else _np.nan
    syl_ratio = (s_hyp / s_ref) if (_np.isfinite(s_ref) and s_ref > 0) else _np.nan

    ref_color_pct = pct(table["Ref Farbdetail"]) if "Ref Farbdetail" in table else _np.nan
    hyp_color_pct = pct(table["Hyp Farbdetail"]) if "Hyp Farbdetail" in table else _np.nan
    ref_move_pct  = pct(table["Ref Bewegung"])  if "Ref Bewegung"  in table else _np.nan
    hyp_move_pct  = pct(table["Hyp Bewegung"])  if "Hyp Bewegung"  in table else _np.nan

    def mean_or_nan(col):
        return float(_np.nanmean(ragas_seg[col])) if ("ragas_seg" in globals() and isinstance(ragas_seg, _pd.DataFrame) and col in ragas_seg) else _np.nan
    rg_sem = mean_or_nan("RAGAS: semantic_similarity")
    rg_fai = mean_or_nan("RAGAS: faithfulness")
    rg_rel = mean_or_nan("RAGAS: answer_relevancy")
    rg_cov = mean_or_nan("RAGAS: coverage")
    rg_con = mean_or_nan("RAGAS: conciseness")
    rg_flu = mean_or_nan("RAGAS: fluency(FRE)")

    recs = []
    if _np.isfinite(len_ratio):
        if len_ratio < 0.9: recs.append("Hyp ist deutlich **kürzer** als Ref → fehlende Inhalte ergänzen.")
        if len_ratio > 1.1: recs.append("Hyp ist **länger** als Ref → Ausschmückungen kürzen.")
    if _np.isfinite(syl_ratio) and syl_ratio > 1.1 and FRE_hyp < 60:
        recs.append("**Viele Silben** & **niedriger FRE** → kürzere Wörter/Strukturen (leichtere Sprache).")
    if _np.isfinite(hyp_color_pct) and _np.isfinite(ref_color_pct) and hyp_color_pct + 5 < ref_color_pct:
        recs.append("**Farbdetails** seltener als in der Ref → visuelle Spezifika konkreter nennen.")
    if _np.isfinite(hyp_move_pct) and _np.isfinite(ref_move_pct) and hyp_move_pct + 5 < ref_move_pct:
        recs.append("**Bewegungsverben** seltener als in der Ref → Handlungen klarer benennen.")
    if _np.isfinite(bleu4_m) and _np.isfinite(bert_m) and bleu4_m < 0.20 and bert_m >= 0.85:
        recs.append("**BLEU-4 niedrig**, **BERTScore hoch** → starke Paraphrasen (Inhalt ähnlich, Wortlaut anders).")
    if _np.isfinite(rL_m) and rL_m < 0.50:
        recs.append("**ROUGE-L F1 < 0.50** → Reihenfolge/Abdeckung prüfen; ggf. Segmente neu ausrichten.")
    if (_np.isfinite(cos_cv) and cos_cv > 40) or (_np.isfinite(rL_cv) and rL_cv > 40):
        recs.append("Hohe **Streuung** (CV%) → Qualität schwankt; schwache Segmente gezielt verbessern.")
    if _np.isfinite(rg_fai) and rg_fai < 0.6:
        recs.append("**Faithfulness** gering → wichtige Referenzinhalte fehlen/entstellt.")
    if _np.isfinite(rg_rel) and rg_rel < 0.6:
        recs.append("**Relevancy** gering → Hyp enthält Nebenaspekte; stärker fokussieren.")
    if not recs:
        recs = ["Keine akuten Auffälligkeiten. Feinjustierung nach Bedarf."]

    # Rubrics kompakt als Markdown
    def _fmt0(x):
        return "—" if (x is None or (isinstance(x, float) and _np.isnan(x))) else f"{float(x):.0f}"
    rubrics_vals = {r: s for r, s, _ in rows_rub}
    rubrics_md = (
        f"- **Stil/Konzision (Wörter):** {_fmt0(rubrics_vals.get('Stil/Konzision (Wörter)'))}/100\n"
        f"- **Stil/Konzision (Silben):** {_fmt0(rubrics_vals.get('Stil/Konzision (Silben)'))}/100\n"
        f"- **Lesehärte (FRE, Hyp):** {_fmt0(rubrics_vals.get('Lesehärte (FRE, Hyp)'))}/100\n"
        f"- **Visuelle Klarheit:** {_fmt0(rubrics_vals.get('Visuelle Klarheit'))}/100\n"
        f"- **Handlungsführung:** {_fmt0(rubrics_vals.get('Handlungsführung'))}/100\n"
        f"- **Inhaltsdeckung/Kohärenz:** {_fmt0(rubrics_vals.get('Inhaltsdeckung/Kohärenz'))}/100\n"
        f"- **Timing-Genauigkeit:** {_fmt0(rubrics_vals.get('Timing-Genauigkeit'))}/100\n"
        f"- **Gesamtnote (gewichtet):** {_fmt0(overall)}/100"
    )

    display(HTML("<h4 style='margin:10px 0 6px 0'>Empfehlungen</h4>"
                 "<ul style='margin:0 0 0 1.2rem'>" +
                 "".join([f"<li>{escape(r)}</li>" for r in recs]) + "</ul>"))

    # — 6) Auto-Report (Markdown) ---------------------------------------------
    cos_mean_v = cos_mean if _np.isfinite(cos_mean) else _np.nan
    sbert_m_v  = sbert_m  if _np.isfinite(sbert_m)  else _np.nan
    bert_m_v   = bert_m   if _np.isfinite(bert_m)   else _np.nan
    rL_m_v     = rL_m     if _np.isfinite(rL_m)     else _np.nan
    bleu4_m_v  = bleu4_m  if _np.isfinite(bleu4_m)  else _np.nan

    report = f"""
# Automatische Interpretation (mit RAGAS & Rubrics)

**Gesamtqualität (Mittelwerte)**  
- TF-IDF Cosine: **{cos_mean_v:.2f}** ({_level(cos_mean_v, THRESHOLDS.get('Cosine (TF-IDF)'))}), Stabilität: {_stability(cos_cv)}  
- SBERT Cosine: **{sbert_m_v:.2f}** ({_level(sbert_m_v, THRESHOLDS.get('SBERT Cosine'))}), Stabilität: {_stability(sbert_cv)}  
- BERTScore F1: **{bert_m_v:.2f}** ({_level(bert_m_v, THRESHOLDS.get('BERTScore F1'))}), Stabilität: {_stability(bert_cv)}  
- ROUGE-L F1: **{rL_m_v:.2f}** ({_level(rL_m_v, THRESHOLDS.get('ROUGE-L F1'))}), Stabilität: {_stability(rL_cv)}  
- BLEU-4: **{bleu4_m_v:.2f}** ({_level(bleu4_m_v, THRESHOLDS.get('BLEU-4'))}), Stabilität: {_stability(bleu4_cv)}

**Timing**  
- Mittlere zeitliche Abweichung (Δ-Mittel): **{(dmean if _np.isfinite(dmean) else _np.nan):.2f} s**  
- Segmente innerhalb ±5 s: **{(share_ok_5s*100.0 if _np.isfinite(share_ok_5s) else _np.nan):.1f}%**  
- Timing-Genauigkeit (1 − Δ/5s, geclippt): **{(timing_acc_mean*100.0 if _np.isfinite(timing_acc_mean) else _np.nan):.1f}%**

**Länge & Lesehärte**  
- Wörter (Ref vs. Hyp): **{(w_ref if _np.isfinite(w_ref) else _np.nan):.0f}** vs. **{(w_hyp if _np.isfinite(w_hyp) else _np.nan):.0f}**  → Verhältnis **{(len_ratio if _np.isfinite(len_ratio) else _np.nan):.2f}**  
- Silben  (Ref vs. Hyp): **{(s_ref if _np.isfinite(s_ref) else _np.nan):.0f}** vs. **{(s_hyp if _np.isfinite(s_hyp) else _np.nan):.0f}**  → Verhältnis **{(syl_ratio if _np.isfinite(syl_ratio) else _np.nan):.2f}**  
- Lesbarkeit Hyp (Flesch–Amstad): **{FRE_hyp:.0f}**/100 (höher = leichter)

**RAGAS (Fallback-äquivalent, 0..1)**  
- Semantic Similarity: **{(rg_sem if _np.isfinite(rg_sem) else _np.nan):.2f}** · Faithfulness: **{(rg_fai if _np.isfinite(rg_fai) else _np.nan):.2f}** · Answer Relevancy: **{(rg_rel if _np.isfinite(rg_rel) else _np.nan):.2f}**  
- Coverage: **{(rg_cov if _np.isfinite(rg_cov) else _np.nan):.2f}** · Conciseness: **{(rg_con if _np.isfinite(rg_con) else _np.nan):.2f}** · Fluency (aus FRE): **{(rg_flu if _np.isfinite(rg_flu) else _np.nan):.2f}**

**Heuristiken (Anteil Segmente mit Merkmal)**  
- Farbdetail: Ref **{(ref_color_pct if _np.isfinite(ref_color_pct) else 0):.0f}%**, Hyp **{(hyp_color_pct if _np.isfinite(hyp_color_pct) else 0):.0f}%**  
- Bewegung:  Ref **{(ref_move_pct  if _np.isfinite(ref_move_pct)  else 0):.0f}%**, Hyp **{(hyp_move_pct  if _np.isfinite(hyp_move_pct)  else 0):.0f}%**

**MDR-Rubrics (0–100, höher besser)**  
{rubrics_md}

**Empfehlungen**  
- {("\n- ").join(recs)}
"""
    globals()["report"] = report  # für Export-Button
    display(Markdown(report))